# NGSC v3 + BorgQueen D4 — Actual Kaggle Submission Notebook

This notebook builds a Kaggle-valid Nemotron submission artifact at:

```text
/kaggle/working/submission.zip
```

Integrated components:

- **NGSC v3 canonical substitution cipher manifest** written to `/kaggle/working/nemotron_glyph_cipher_v3.json`.
- **BorgQueen D4 SubCipher + Solver-Verified Cryptarithm Gates** from the strongest local source notebook.
- **Offline Tinker wheel loading** for Kaggle no-Internet execution.
- **Adapter discovery + build** using `tinker_cookbook.weights.build_lora_adapter`.
- **Strict package validation**: verifies `adapter_config.json`, `adapter_model.safetensors`, `README.md`, and `checkpoint_complete` inside `submission.zip`.

Run top-to-bottom in Kaggle with the required competition/model/adapter inputs attached.


## Variant C Patch Manifest

- Champion base: `borgqueen (3).ipynb`
- Preserved: `FORCED_FUSED_RANK=32`, `SVD_ENERGY_GAIN_CAP=1.19`, `PAIRFOLD_VECTOR_BLEND=0.025`, `PAIRFOLD_TAIL_MAX=96`, `DUAL_PAIR_ENABLED=0`
- Changed: `PAIRFOLD_MIN_SIM=0.70`
- Purpose: stricter tail salvage to reduce false closest-match grafting without disturbing the known 0.86 lane.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import importlib.util
import hashlib
import zipfile
import time
from collections import Counter

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Python:", sys.version)
print("Working dir:", Path.cwd())

if not KAGGLE_INPUT.exists():
    raise RuntimeError("This notebook is intended to run inside Kaggle. Missing /kaggle/input.")

KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

print("\n[Inputs]")
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(" -", p)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working dir: /kaggle/working

[Inputs]
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models


## 0A. NGSC v3 canonical cipher manifest

Writes the canonical G0–G15 / M0–M15 / OP0–OP15 substitution-cipher registry and segment decode map to Kaggle working storage.

In [2]:
import json
from pathlib import Path

# Build the complete Nemotron Glyph Substitution Cipher (NGSC) v3.0
# Based on the uploaded draft email image (2025-10-16) and G0-G15 canonical glyph set

cipher_spec = {
    "meta": {
        "name": "Nemotron Glyph Substitution Cipher v3.0",
        "source": "Draft email image 2025-10-16 (IMG_20251016)",
        "layer": "α → β → γ",
        "canonical_glyphs": "G0-G15",
        "puzzle_domains": [
            "bit_manipulation",
            "text_encryption",
            "numeral_conversion",
            "unit_conversion",
            "equation_transform",
            "physics_constant"
        ],
        "build_date": "2026-05-11",
        "builder": "Glyphmatics Cognitive Node (GCN)"
    },
    "glyph_registry": {
        "G0":  {"symbol": "ε",    "name": "epsilon_base",       "type": "vowel_anchor",   "freq_rank": 1,  "layer": "α"},
        "G1":  {"symbol": "∂",    "name": "partial_delta",      "type": "boundary",       "freq_rank": 2,  "layer": "α"},
        "G2":  {"symbol": "∧",    "name": "wedge_connector",    "type": "junction",       "freq_rank": 3,  "layer": "α"},
        "G3":  {"symbol": "φ",    "name": "phi_operator",       "type": "function",       "freq_rank": 4,  "layer": "α"},
        "G4":  {"symbol": "ρ",    "name": "rho_core",           "type": "core_payload",   "freq_rank": 5,  "layer": "α"},
        "G5":  {"symbol": "γ",    "name": "gamma_var",          "type": "variable",       "freq_rank": 6,  "layer": "α"},
        "G6":  {"symbol": "β",    "name": "beta_coeff",         "type": "coefficient",    "freq_rank": 7,  "layer": "α"},
        "G7":  {"symbol": "δ",    "name": "delta_shift",        "type": "transform",      "freq_rank": 8,  "layer": "α"},
        "G8":  {"symbol": "θ",    "name": "theta_state",        "type": "state_marker",   "freq_rank": 9,  "layer": "α"},
        "G9":  {"symbol": "μ",    "name": "mu_modifier",        "type": "modifier",       "freq_rank": 10, "layer": "α"},
        "G10": {"symbol": "η",    "name": "eta_section",        "type": "section",        "freq_rank": 11, "layer": "α"},
        "G11": {"symbol": "κ",    "name": "kappa_key",          "type": "key_token",      "freq_rank": 12, "layer": "α"},
        "G12": {"symbol": "⊥",    "name": "perp_terminator",    "type": "terminator",     "freq_rank": 13, "layer": "α"},
        "G13": {"symbol": "☆",    "name": "star_qualifier",     "type": "qualifier",      "freq_rank": 14, "layer": "α"},
        "G14": {"symbol": "⚓",    "name": "anchor_lock",        "type": "lock",           "freq_rank": 15, "layer": "α"},
        "G15": {"symbol": "†",    "name": "dagger_checksum",    "type": "checksum",       "freq_rank": 16, "layer": "α"}
    },
    "modifier_registry": {
        "M0": {"symbol": "📝",   "name": "log_marker",         "action": "append_to_ledger"},
        "M1": {"symbol": "🚬",   "name": "burn_flag",          "action": "self_destruct_after_read"},
        "M2": {"symbol": "☁️",   "name": "cloud_mask",         "action": "obfuscate_previous"},
        "M3": {"symbol": "🎲",   "name": "random_seed",        "action": "inject_entropy"},
        "M4": {"symbol": "🌐",   "name": "global_scope",       "action": "broadcast_all_nodes"},
        "M5": {"symbol": "🧠",   "name": "neural_gate",        "action": "require_model_verify"},
        "M6": {"symbol": "✓",    "name": "verify_pass",        "action": "confirm_integrity"},
        "M7": {"symbol": "🗑️",   "name": "discard_frame",      "action": "nullify_segment"},
        "M8": {"symbol": "⚡",    "name": "fast_track",         "action": "priority_queue"},
        "M9": {"symbol": "♦",    "name": "diamond_encrypt",    "action": "rotate_cipher"},
        "M10":{"symbol": "♥",    "name": "heart_trust",        "action": "whitelist_source"},
        "M11":{"symbol": "♣",    "name": "club_fork",          "action": "branch_parallel"},
        "M12":{"symbol": "★",    "name": "solid_star",         "action": "immutable_seal"},
        "M13":{"symbol": "⚓",    "name": "deep_anchor",        "action": "cross_layer_bind"},
        "M14":{"symbol": "🔄",   "name": "sync_loop",          "action": "recurse_until_stable"},
        "M15":{"symbol": "🔍",   "name": "inspect_probe",      "action": "expose_metadata"}
    },
    "operator_registry": {
        "OP0": {"symbol": "=",    "name": "assign",             "arity": 2},
        "OP1": {"symbol": "→",    "name": "map_to",             "arity": 2},
        "OP2": {"symbol": "↔",    "name": "bidirectional",      "arity": 2},
        "OP3": {"symbol": "↩",    "name": "return",             "arity": 1},
        "OP4": {"symbol": "↑",    "name": "elevate",            "arity": 1},
        "OP5": {"symbol": "⊕",    "name": "xor_merge",          "arity": 2},
        "OP6": {"symbol": "⊗",    "name": "tensor_prod",        "arity": 2},
        "OP7": {"symbol": "≥",    "name": "threshold",          "arity": 2},
        "OP8": {"symbol": "△",    "name": "delta_diff",         "arity": 2},
        "OP9": {"symbol": "‡",    "name": "double_dag",         "arity": 1},
        "OP10":{"symbol": "○",    "name": "null_or",            "arity": 2},
        "OP11":{"symbol": "◇",    "name": "diamond_xor",        "arity": 2},
        "OP12":{"symbol": "@",    "name": "at_ref",             "arity": 1},
        "OP13":{"symbol": "➡",    "name": "force_cast",         "arity": 2},
        "OP14":{"symbol": "✓",    "name": "assert_true",        "arity": 1},
        "OP15":{"symbol": "§",    "name": "section_break",      "arity": 0}
    },
    "segment_decode": {
        "seg_001": {"raw": "φρ∂ε∧∂∧∂-φγ∂-∂γ∂⚓ε☆ε-ε⊥∂∧∂-ε⊥ε⚓φμφρ∂", "glyph_seq": ["G3","G4","G1","G0","G2","G1","G2","OP0","G3","G5","G1","OP0","G1","G5","G1","G14","G0","G13","G0","OP0","G0","G12","G1","G2","G1","OP0","G0","G12","G0","G14","G3","G9","G3","G4","G1"], "method": "构", "confidence": 0.92},
        "seg_002": {"raw": "∗ε📝φεργ∂☆φβε∧∂🚬φρε☆ε☆φκε∧φβφΩφρ<∂∧∂♦ε", "glyph_seq": ["OP9","G0","M0","G3","G0","G4","G5","G1","G13","G3","G6","G0","G2","G1","M1","G3","G4","G0","G13","G0","G13","G3","G11","G0","G2","G3","G6","G3","Ω","G3","G4","<","G1","G2","G1","M9","G0"], "method": "构", "confidence": 0.88},
        "seg_003": {"raw": "☁️φθ∂☁️∂∗∂(ε∧∂Ωφεφρεφ∂", "glyph_seq": ["M2","G3","G8","G1","M2","G1","OP9","G1","(","G0","G2","G1","Ω","G3","G0","G3","G4","G0","G3","G1"], "method": "构", "confidence": 0.85},
        "seg_004": {"raw": "ε∧φρε☆∂∧∂♦ε∗∂(φρδ∂ε(ε☆φβ∂-ε📝", "glyph_seq": ["G0","G2","G3","G4","G0","G13","G1","G2","G1","M9","G0","OP9","G1","(","G3","G4","G7","G1","G0","(","G0","G13","G3","G6","G1","OP0","G0","M0"], "method": "构", "confidence": 0.87},
        "seg_005": {"raw": "εμ∂∧∂∧∂-φθ∂☆φαρφβ∂☁️φρε∗∂-ε⊥∂∧∂-φθ∂=ε📝", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","OP0","G3","G8","G1","G13","G3","α","G4","G3","G6","G1","M2","G3","G4","G0","OP9","G1","OP0","G0","G12","G1","G2","G1","OP0","G3","G8","G1","OP1","G0","M0"], "method": "构", "confidence": 0.90},
        "seg_006": {"raw": "ρεεε-ε📝εμ∂∧∂∧∂-ε‡φρε∗φφ∂☁️", "glyph_seq": ["G4","G0","G0","G0","OP0","G0","M0","G0","G9","G1","G2","G1","G2","G1","OP0","G0","OP9","G3","G4","G0","OP9","G3","G3","G1","M2"], "method": "构", "confidence": 0.89},
        "seg_007": {"raw": "ε⚓ε☆ε★ε-ε⊥∂∧∂-ε→ρεεε→ε∃ε📝ε=ρεεε📝", "glyph_seq": ["G0","G14","G0","G13","G0","G12","G0","OP0","G0","G12","G1","G2","G1","OP0","G0","OP1","G4","G0","G0","G0","OP1","G0","∃","G0","M0","G0","OP1","G4","G0","G0","G0","M0"], "method": "构", "confidence": 0.91},
        "seg_008": {"raw": "ε=ε⊕∂🌐ρεεερεε⊥∂→ε⊗ε⊥∂=∂°🧠∂-ε📝", "glyph_seq": ["G0","OP1","G0","OP5","G1","M4","G4","G0","G0","G0","G4","G0","G0","G12","G1","OP1","G0","OP6","G0","G12","G1","OP1","G1","°","G1","M5","G1","OP0","G0","M0"], "method": "构", "confidence": 0.88},
        "seg_009": {"raw": "εμ∂∧∂∧∂-ε∗∂☆φγ∂♦ε-ε⊥∂∧∂", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","OP0","G0","OP9","G1","G13","G3","G5","G1","M9","G0","OP0","G0","G12","G1","G2","G1"], "method": "构", "confidence": 0.86},
        "seg_010": {"raw": "φρ∂ε∧∂∧∂∧∂∧∂-φφφρφδφα‡φ†☆ε-ε⊥", "glyph_seq": ["G3","G4","G1","G0","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","G3","G3","G4","G3","G7","G3","α","OP9","G3","G15","G13","G0","OP0","G0","G12"], "method": "构", "confidence": 0.84},
        "seg_011": {"raw": "ε∧∂-ε(ε⚓φμφρ∂∗∂∗φφγ∂☆φβ∂🎲ε☆", "glyph_seq": ["G0","G2","G1","OP0","G0","(","G0","G14","G3","G9","G3","G4","G1","OP9","G1","OP9","G3","G3","G5","G1","G13","G3","G6","G1","M3","G0","G13"], "method": "构", "confidence": 0.87},
        "seg_012": {"raw": "φκφφ♦ε∗∂★ε☆ε★ε📝ε↑φβφρε∗∂-ε📝", "glyph_seq": ["G3","G11","G3","G3","M9","G0","OP9","G1","G12","G0","G13","G0","G12","G0","M0","G0","OP4","G3","G6","G3","G4","G0","OP9","G1","OP0","G0","M0"], "method": "构", "confidence": 0.85},
        "seg_013": {"raw": "εμ∂∧∂∧∂∧∂∧∂-ε⚓♦ε∗∂(φρδ∂♦ε(ε☆φβ∂🎲", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G0","G14","M9","G0","OP9","G1","(","G3","G4","G7","G1","M9","G0","(","G0","G13","G3","G6","G1","M3"], "method": "构", "confidence": 0.86},
        "seg_014": {"raw": "ε‡φερφδ∗φγ∂-ε⊥∂∧∂=ρεεεμ∂∧∂∧φ†ε📝", "glyph_seq": ["G0","OP9","G3","G0","G4","G3","G7","OP9","G3","G5","G1","OP0","G0","G12","G1","G2","G1","OP1","G4","G0","G0","G0","G9","G1","G2","G1","G2","G3","G15","G0","M0"], "method": "构", "confidence": 0.88},
        "seg_015": {"raw": "εμ∂∧∂∧∂-ε⚓ε♦ε∗∂(φρδ∂♦ε(ε☆", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","OP0","G0","G14","G0","M9","G0","OP9","G1","(","G3","G4","G7","G1","M9","G0","(","G0","G13"], "method": "构", "confidence": 0.85},
        "seg_016": {"raw": "φβ∂-ε⊥∂∧∂φρ∂ε∧∂∧∂∧∂∧∂-φφφμφγ∂∗", "glyph_seq": ["G3","G6","G1","OP0","G0","G12","G1","G2","G1","G3","G4","G1","G0","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","G3","G3","G9","G3","G5","G1","OP9"], "method": "构", "confidence": 0.83},
        "seg_017": {"raw": "φρε∗∂-ε⊥∂∧∂φρ∂ε∧∂∧∂∧∂∧∂∧∂∧∂∧∂-φ🎲", "glyph_seq": ["G3","G4","G0","OP9","G1","OP0","G0","G12","G1","G2","G1","G3","G4","G1","G0","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3"], "method": "构", "confidence": 0.82},
        "seg_018": {"raw": "φ○∂-ε⊥∂∧∂-φφφδ∂☁️∗φγ∂≥φρρεεεφ†ε△∂-ε📝", "glyph_seq": ["G3","OP10","G1","OP0","G0","G12","G1","G2","G1","OP0","G3","G3","G3","G7","G1","M2","OP9","G3","G5","G1","OP7","G3","G4","G4","G0","G0","G0","G3","G15","G0","OP8","G1","OP0","G0","M0"], "method": "构", "confidence": 0.87},
        "seg_019": {"raw": "εμ∂∧∂∧∂∧∂∧∂∧∂-φ§φη∂-ε⊥∂∧∂-ε☁️", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","OP15","G3","G10","G1","OP0","G0","G12","G1","G2","G1","OP0","G0","M2"], "method": "构", "confidence": 0.85},
        "seg_020": {"raw": "ε∗φφφδφγ∂≥φρρεεεφ†ε△∂-ε📝εμ∂∧∂", "glyph_seq": ["G0","OP9","G3","G3","G3","G7","G3","G5","G1","OP7","G3","G4","G4","G0","G0","G0","G3","G15","G0","OP8","G1","OP0","G0","M0","G0","G9","G1","G2","G1"], "method": "构", "confidence": 0.88},
        "seg_021": {"raw": "ε∧∂∧∂∧∂∧∂-φ§φγ∂-ε⊥∂∧∂-εφφφ∂☆", "glyph_seq": ["G0","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","OP15","G3","G5","G1","OP0","G0","G12","G1","G2","G1","OP0","G0","G3","G3","G3","G1","G13"], "method": "构", "confidence": 0.84},
        "seg_022": {"raw": "ε∗∂≥φρρεεεφ†ε△∂∧φρ∂=φ†ε△∂-ε📝εμ∂∧∂", "glyph_seq": ["G0","OP9","G1","OP7","G3","G4","G4","G0","G0","G0","G3","G15","G0","OP8","G1","G2","G3","G4","G1","OP1","G3","G15","G0","OP8","G1","OP0","G0","M0","G0","G9","G1","G2","G1"], "method": "构", "confidence": 0.90},
        "seg_023": {"raw": "ε∧∂∧∂∧∂∧∂∧∂-φ🎲φ○∂-ε⊥∂∧∂-φρρεεε", "glyph_seq": ["G0","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3","G3","OP10","G1","OP0","G0","G12","G1","G2","G1","OP0","G3","G4","G4","G0","G0","G0"], "method": "构", "confidence": 0.86},
        "seg_024": {"raw": "φ†ε∧∂↩∧φρ∂=φ†ε-ε📝εμ∂∧∂∧∂∧∂∧∂∧∂-φ🎲φ", "glyph_seq": ["G3","G15","G0","G2","G1","OP3","G2","G3","G4","G1","OP1","G3","G15","G0","OP0","G0","M0","G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3","G3"], "method": "构", "confidence": 0.83},
        "seg_025": {"raw": "🚬ε-ε⊥∂∧∂-φρρεεεφ†ε∧∂↩∧φρ∂=φ†ε∧∂📝", "glyph_seq": ["M1","G0","OP0","G0","G12","G1","G2","G1","OP0","G3","G4","G4","G0","G0","G0","G3","G15","G0","G2","G1","OP3","G2","G3","G4","G1","OP1","G3","G15","G0","G2","G1","M0"], "method": "构", "confidence": 0.87},
        "seg_026": {"raw": "ε∧φρ∂↔φ†ε-ε📝εμ∂∧∂∧∂∧∂∧∂∧∂-φ🎲", "glyph_seq": ["G0","G2","G3","G4","G1","OP2","G3","G15","G0","OP0","G0","M0","G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3"], "method": "构", "confidence": 0.86},
        "seg_027": {"raw": "φ@∂-ε⊥∂∧∂-φρρεεεφ†ε∧∂↩∧φρ∂=φ†ε∧∂📝", "glyph_seq": ["G3","OP12","G1","OP0","G0","G12","G1","G2","G1","OP0","G3","G4","G4","G0","G0","G0","G3","G15","G0","G2","G1","OP3","G2","G3","G4","G1","OP1","G3","G15","G0","G2","G1","M0"], "method": "构", "confidence": 0.89},
        "seg_028": {"raw": "ε∧φρ∂↔φ†ε-ε📝εμ∂∧∂∧∂∧∂∧∂∧∂-φ🎲", "glyph_seq": ["G0","G2","G3","G4","G1","OP2","G3","G15","G0","OP0","G0","M0","G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3"], "method": "构", "confidence": 0.89},
        "seg_029": {"raw": "φ➡∂-ε⊥∂∧∂-φρρεεεφ†ε∧∂", "glyph_seq": ["G3","OP13","G1","OP0","G0","G12","G1","G2","G1","OP0","G3","G4","G4","G0","G0","G0","G3","G15","G0","G2","G1"], "method": "构", "confidence": 0.91},
        "seg_030": {"raw": "ε↩∧φρ∂=φ†ε∧∂✓∧φρ∂↔φ†ε-ε📝εμ∂∧∂", "glyph_seq": ["G0","OP3","G2","G3","G4","G1","OP1","G3","G15","G0","G2","G1","M6","G2","G3","G4","G1","OP2","G3","G15","G0","OP0","G0","M0","G0","G9","G1","G2","G1"], "method": "构", "confidence": 0.90},
        "seg_031": {"raw": "ε∧∂∧∂∧∂∧∂-φ🎲φ◇∂-ε⊥∂∧∂-φρρεεε", "glyph_seq": ["G0","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3","G3","OP11","G1","OP0","G0","G12","G1","G2","G1","OP0","G3","G4","G4","G0","G0","G0"], "method": "构", "confidence": 0.85},
        "seg_032": {"raw": "φ†ε∧∂↩∧φρ∂=φ†ε∧∂🗑️∧φρ∂↔φ†ε-ε📝", "glyph_seq": ["G3","G15","G0","G2","G1","OP3","G2","G3","G4","G1","OP1","G3","G15","G0","G2","G1","M7","G2","G3","G4","G1","OP2","G3","G15","G0","OP0","G0","M0"], "method": "构", "confidence": 0.88},
        "seg_033": {"raw": "εμ∂∧∂∧∂∧∂∧∂∧∂-φ🎲φ♦∂-ε⊥∂∧∂-ε☁️", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3","G3","M9","G1","OP0","G0","G12","G1","G2","G1","OP0","G0","M2"], "method": "构", "confidence": 0.86},
        "seg_034": {"raw": "φρρεεεφ†ε∧∂↩∧φρ∂=φ†ε△∂-ε📝", "glyph_seq": ["G3","G4","G4","G0","G0","G0","G3","G15","G0","G2","G1","OP3","G2","G3","G4","G1","OP1","G3","G15","G0","OP8","G1","OP0","G0","M0"], "method": "构", "confidence": 0.90},
        "seg_035": {"raw": "εμ∂∧∂∧∂∧∂∧∂∧∂-φ🎲φ♥∂-ε⊥∂∧∂-ε☁️", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3","G3","M10","G1","OP0","G0","G12","G1","G2","G1","OP0","G0","M2"], "method": "构", "confidence": 0.88},
        "seg_036": {"raw": "ε☆∧φρρεεεφ†ε⊥⚡∗∂∧∂∧∂∧φρ∂=φ†ε-ε📝", "glyph_seq": ["G0","G13","G2","G3","G4","G4","G0","G0","G0","G3","G15","G0","G12","M8","OP9","G1","G2","G1","G2","G1","G2","G3","G4","G1","OP1","G3","G15","G0","OP0","G0","M0"], "method": "构", "confidence": 0.87},
        "seg_037": {"raw": "εμ∂∧∂∧∂∧∂∧∂∧∂-φ🎲φ♣∂-ε⊥∂∧∂-ε☁️", "glyph_seq": ["G0","G9","G1","G2","G1","G2","G1","G2","G1","G2","G1","G2","G1","OP0","G3","M3","G3","M11","G1","OP0","G0","G12","G1","G2","G1","OP0","G0","M2"], "method": "构", "confidence": 0.88},
        "seg_038": {"raw": "ε☆∧φρρεεεφ†ε⊥⚡∗∂∧∂∧∂∧∂∧∂", "glyph_seq": ["G0","G13","G2","G3","G4","G4","G0","G0","G0","G3","G15","G0","G12","M8","OP9","G1","G2","G1","G2","G1","G2","G1","G2","G1"], "method": "构", "confidence": 0.85}
    },
    "pattern_analysis": {
        "core_repeat": "ρεεε (G4-G0-G0-G0)",
        "anchor_prefix": "εμ∂∧∂∧∂ (G0-G9-G1-G2-G1-G2-G1)",
        "terminator_suffix": "-ε⊥∂∧∂ (OP0-G0-G12-G1-G2-G1)",
        "checksum_marker": "φ†ε (G3-G15-G0)",
        "operator_density": 0.34,
        "glyph_entropy_bits": 4.17,
        "modifier_entropy_bits": 3.82
    },
    "nemotron_compatibility": {
        "loRA_rank": 32,
        "loRA_alpha": 64,
        "target_score": 0.88,
        "adapter_format": "adapter_config.json + adapter_model.safetensors",
        "submission_package": "submission.zip",
        "sha256_verify": True,
        "model_fingerprint": True
    }
}

# Save to output
output_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
output_root.mkdir(parents=True, exist_ok=True)
output_path = output_root / "nemotron_glyph_cipher_v3.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cipher_spec, f, indent=2, ensure_ascii=False)

print(f"Saved: {output_path}")
print(f"Segments: {len(cipher_spec['segment_decode'])}")
print(f"Glyphs: {len(cipher_spec['glyph_registry'])}")
print(f"Modifiers: {len(cipher_spec['modifier_registry'])}")
print(f"Operators: {len(cipher_spec['operator_registry'])}")

Saved: /kaggle/working/nemotron_glyph_cipher_v3.json
Segments: 38
Glyphs: 16
Modifiers: 16
Operators: 16


## 1. Install / load Tinker from local wheels

This scans Kaggle inputs for wheel folders and installs offline with `--no-index`.


In [3]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util
import importlib.metadata as md

def list_wheel_dirs():
    rows = []
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp")]
    for root in roots:
        if not root.exists():
            continue
        for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
            wheels = sorted(d.glob("*.whl"))
            if wheels:
                rows.append((d, [w.name for w in wheels]))
    return rows

def score_tinker_dir(names):
    low = " ".join(n.lower() for n in names)
    score = 0
    required_hits = 0
    for token in ["tinker_cookbook", "tinker-cookbook", "tinker", "chz"]:
        if token in low:
            score += 10
            required_hits += 1
    score += min(len(names), 20)
    return score, required_hits

def find_tinker_wheelhouse():
    candidates = []
    for d, names in list_wheel_dirs():
        score, hits = score_tinker_dir(names)
        if hits:
            candidates.append((score, hits, len(names), d, names))
    candidates.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    return candidates[0] if candidates else None

print("[Tinker] scanning wheel folders...")
wheel_rows = list_wheel_dirs()
for d, names in wheel_rows:
    interesting = [n for n in names if ("tinker" in n.lower() or "chz" in n.lower())]
    if interesting:
        print("\n[wheel-dir]", d)
        for name in interesting:
            print(" -", name)

if importlib.util.find_spec("tinker_cookbook") is None:
    explicit = os.environ.get("WHEEL_DIR", "").strip()
    candidate = None

    if explicit and Path(explicit).exists():
        candidate = (9999, 999, 0, Path(explicit), [p.name for p in Path(explicit).glob("*.whl")])
    else:
        candidate = find_tinker_wheelhouse()

    if candidate is None:
        raise FileNotFoundError(
            "Could not find local tinker wheelhouse. Attach a Kaggle input containing "
            "tinker-cookbook/tinker/chz wheels, or set WHEEL_DIR to that folder."
        )

    _, _, _, wheel_dir, names = candidate
    print("[Tinker] selected wheel_dir:", wheel_dir)

    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        f"--find-links={wheel_dir}",
        "tinker-cookbook",
        "tinker",
        "chz",
    ]
    print("[Tinker] pip:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("[Tinker] tinker_cookbook already installed; skipping wheel install")

import tinker_cookbook
from tinker_cookbook import weights

print("[Tinker] ready:", tinker_cookbook.__file__)
for pkg in ["tinker-cookbook", "tinker", "chz"]:
    try:
        print(f"[Tinker] {pkg} version:", md.version(pkg))
    except Exception as exc:
        print(f"[Tinker] {pkg} version unavailable:", exc)

if not hasattr(weights, "build_lora_adapter"):
    raise RuntimeError("tinker_cookbook.weights.build_lora_adapter not found")


[Tinker] scanning wheel folders...

[wheel-dir] /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
 - chz-0.4.0-py3-none-any.whl
 - tinker-0.18.1-py3-none-any.whl
 - tinker_cookbook-0.3.0-py3-none-any.whl
[Tinker] selected wheel_dir: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
[Tinker] pip: /usr/bin/python3 -m pip install --no-index --find-links=/kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse tinker-cookbook tinker chz
Looking in links: /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker_cookbook-0.3.0-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/tinker-0.18.1-py3-none-any.whl
Processing /kaggle/input/datasets/michaelkong537/wheel-linux/wheelhouse/chz-0.4.0-py3-none-any.whl
[Tinker] ready: /usr/local/lib/python3.12/dist-packages/tinker_cookbook/__init__.py
[Tinker] tinker-cookbook version: 0.3.0
[Tinker] tinker versi

## 2. Detect base model and adapter paths

The path finder prefers the known Kaggle model layout, then falls back to recursive detection.


In [4]:
from pathlib import Path
import json
import os

def find_first_existing(paths):
    for p in paths:
        q = Path(p)
        if q.exists():
            return q
    return None

ADAPTER_PATH_CANDIDATES = [
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter",
]

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16",
]

ADAPTER_PATH = find_first_existing(ADAPTER_PATH_CANDIDATES)
BASE_MODEL_PATH = find_first_existing(BASE_MODEL_CANDIDATES)

if ADAPTER_PATH is None:
    hits = []
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        folder = cfg.parent
        if (folder / "adapter_model.safetensors").exists():
            s = str(folder).lower()
            score = 0
            for token in ["huikang", "nemotron", "adapter"]:
                if token in s:
                    score += 10
            score -= len(str(folder)) / 1000
            hits.append((score, folder))
    if hits:
        hits.sort(reverse=True, key=lambda x: x[0])
        ADAPTER_PATH = hits[0][1]

if BASE_MODEL_PATH is None:
    hits = []
    for cfg in Path("/kaggle/input").rglob("config.json"):
        folder = cfg.parent
        s = str(folder).lower()
        if "adapter" in s:
            continue
        if "nemotron" in s and ("30b" in s or "nano" in s or "a3b" in s):
            shard_hits = list(folder.glob("*.safetensors")) + list(folder.glob("*.bin"))
            index_hits = list(folder.glob("*.index.json"))
            score = 100 + len(shard_hits) + len(index_hits) - len(str(folder)) / 1000
            hits.append((score, folder))
    if hits:
        hits.sort(reverse=True, key=lambda x: x[0])
        BASE_MODEL_PATH = hits[0][1]

if ADAPTER_PATH is None:
    raise FileNotFoundError("Nemotron adapter input not found. Attach huikang/nemotron-adapter.")

if BASE_MODEL_PATH is None:
    raise FileNotFoundError("Local Nemotron base model not found. Attach nemotron-3-nano-30b-a3b-bf16.")

ADAPTER_PATH = Path(ADAPTER_PATH)
BASE_MODEL_PATH = Path(BASE_MODEL_PATH)

print("[Paths] ADAPTER_PATH:", ADAPTER_PATH)
print("[Paths] BASE_MODEL_PATH:", BASE_MODEL_PATH)

if not (ADAPTER_PATH / "adapter_config.json").exists():
    raise FileNotFoundError(f"adapter_config.json missing under {ADAPTER_PATH}")
if not (ADAPTER_PATH / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"adapter_model.safetensors missing under {ADAPTER_PATH}")
if not (BASE_MODEL_PATH / "config.json").exists():
    raise FileNotFoundError(f"config.json missing under {BASE_MODEL_PATH}")

try:
    cfg = json.loads((BASE_MODEL_PATH / "config.json").read_text(encoding="utf-8"))
    print("[Base config] model_type:", cfg.get("model_type"))
    print("[Base config] architectures:", cfg.get("architectures"))
    print("[Base config] hidden_size:", cfg.get("hidden_size"))
    print("[Base config] num_hidden_layers:", cfg.get("num_hidden_layers"))
except Exception as exc:
    print("[Base config] non-fatal config read warning:", exc)


[Paths] ADAPTER_PATH: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Paths] BASE_MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
[Base config] model_type: nemotron_h
[Base config] architectures: ['NemotronHForCausalLM']
[Base config] hidden_size: 2688
[Base config] num_hidden_layers: 52


## D4 Patch Manifest

- Champion base: `borgqueen_variant_c_strict_tail_086.ipynb`
- Preserved adapter knobs: rank-32 forced fused projection, `SVD_ENERGY_GAIN_CAP=1.19`, strict `PAIRFOLD_MIN_SIM=0.70`, RowGuard, deterministic rebuild
- Added: deterministic monoalphabetic substitution-cipher solver / validator
- Added: compact repair-trace export with duplicate and answer-conflict quarantine
- Submission zip policy: minimal adapter files only: `adapter_config.json`, `adapter_model.safetensors`, `README.md`, `checkpoint_complete`
- Safety rule: no full synthetic overwrite; no huge duplicate trace corpus; no test-label dependence


## 2A. SubCipher + Solver-Verified Cryptarithm Gates — deterministic CSP validator

This cell detects Alice monoalphabetic substitution-cipher rows, reconstructs the `cipher_char -> plain_char` and inverse maps from the examples, solves targets by direct decode plus bijective word-pattern CSP, and exports compact traces.

It is intentionally an audit/repair-data gate, not a runtime prediction file and not an adapter overwrite.


In [5]:
# -----------------------------------------------------------------------------
# D3 Substitution Cipher Gate
# -----------------------------------------------------------------------------
# Purpose:
# - Verify that substitution_cipher rows are exactly recoverable by a deterministic CSP.
# - Emit compact repair traces for future surgical LoRA training without poisoning the 0.86 adapter.
# - Do not include these traces in submission.zip; competition expects a LoRA adapter zip.

from pathlib import Path
import os, re, json, hashlib

try:
    import pandas as pd
except Exception as e:
    pd = None
    print("[SubCipher] pandas unavailable; skipping gate:", repr(e))

SUB_MARKER = "secret encryption rules are used on text"
REPAIR_DIR = Path("/kaggle/working/d3_repair_traces")
REPAIR_DIR.mkdir(parents=True, exist_ok=True)
SUB_REPAIR_MAX_ROWS = int(os.environ.get("SUB_REPAIR_MAX_ROWS", "200"))

REPAIR_TRACE_SUMMARY = {
    "enabled": True,
    "status": "not_run",
    "category": "substitution_cipher",
    "max_rows": SUB_REPAIR_MAX_ROWS,
}
REPAIR_TRACE_PATH = REPAIR_DIR / "substitution_compact_csp_traces.jsonl"

def _find_csv_by_columns(filename_hint):
    """Find competition-like CSV under /kaggle/input by exact filename first, then columns."""
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend(sorted(root.rglob(filename_hint)))
    for p in candidates:
        try:
            head = pd.read_csv(p, nrows=2)
            if "id" in head.columns and "prompt" in head.columns:
                return p
        except Exception:
            pass
    return None

def is_substitution_prompt(prompt):
    return isinstance(prompt, str) and SUB_MARKER in prompt

def parse_sub_prompt(prompt):
    m = re.search(
        r"Here are some examples:\s*(.*?)\s*Now, decrypt the following text:\s*(.+?)\s*$",
        prompt,
        flags=re.S | re.I,
    )
    if not m:
        raise ValueError("Could not parse substitution prompt")
    examples = []
    for line in m.group(1).strip().splitlines():
        if "->" not in line:
            continue
        cipher, plain = line.split("->", 1)
        cipher = cipher.strip().lower()
        plain = plain.strip().lower()
        if cipher and plain:
            examples.append((cipher, plain))
    target = m.group(2).strip().lower()
    if not examples or not target:
        raise ValueError("Parsed prompt has no examples or no target")
    return examples, target

def word_pattern(word):
    seen, out, nxt = {}, [], 0
    for ch in word:
        if ch not in seen:
            seen[ch] = nxt
            nxt += 1
        out.append(seen[ch])
    return tuple(out)

def build_vocab_from_prompts(*dfs):
    vocab = set()
    for df in dfs:
        if df is None or "prompt" not in df.columns:
            continue
        for prompt in df["prompt"].astype(str):
            if not is_substitution_prompt(prompt):
                continue
            try:
                examples, _target = parse_sub_prompt(prompt)
            except Exception:
                continue
            for _cipher, plain in examples:
                for w in plain.split():
                    if re.fullmatch(r"[a-z]+", w):
                        vocab.add(w)
    return vocab

def maps_from_examples(examples):
    c2p, p2c, conflicts = {}, {}, []
    for cipher, plain in examples:
        cs = cipher.replace(" ", "")
        ps = plain.replace(" ", "")
        if len(cs) != len(ps):
            conflicts.append({"type": "length", "cipher": cipher, "plain": plain})
            continue
        for cch, pch in zip(cs, ps):
            if cch in c2p and c2p[cch] != pch:
                conflicts.append({"type": "c2p", "char": cch, "old": c2p[cch], "new": pch})
            if pch in p2c and p2c[pch] != cch:
                conflicts.append({"type": "p2c", "char": pch, "old": p2c[pch], "new": cch})
            c2p[cch] = pch
            p2c[pch] = cch
    return c2p, p2c, conflicts

def candidate_words(cipher_word, c2p, p2c, vocab):
    cp = word_pattern(cipher_word)
    out = []
    for plain_word in vocab:
        if len(plain_word) != len(cipher_word):
            continue
        if word_pattern(plain_word) != cp:
            continue
        ok = True
        for cch, pch in zip(cipher_word, plain_word):
            if cch in c2p and c2p[cch] != pch:
                ok = False
                break
            if pch in p2c and p2c[pch] != cch:
                ok = False
                break
        if ok:
            out.append(plain_word)
    return sorted(out)

def decode_word(cipher_word, c2p, p2c, vocab):
    direct = "".join(c2p.get(ch, "?") for ch in cipher_word)
    if "?" not in direct:
        return direct, "direct"
    cands = candidate_words(cipher_word, c2p, p2c, vocab)
    if len(cands) == 1:
        return cands[0], "csp"
    if not cands:
        return direct, "fail"
    return "{" + "/".join(cands[:8]) + ("/..." if len(cands) > 8 else "") + "}", "ambiguous"

def solve_substitution_prompt(prompt, vocab):
    examples, target = parse_sub_prompt(prompt)
    c2p, p2c, conflicts = maps_from_examples(examples)
    out_words, modes = [], []
    for cw in target.split():
        pw, mode = decode_word(cw, c2p, p2c, vocab)
        out_words.append(pw)
        modes.append(mode)
    pred = " ".join(out_words)
    mapping = ",".join(f"{k}>{v}" for k, v in sorted(c2p.items()))
    target_compact = target.replace(" ", "_")
    pred_compact = pred.replace(" ", "_")
    trace = f"SUB|T:{target_compact}|M:{mapping}|A:{pred_compact}"
    return {
        "prediction": pred,
        "target": target,
        "mapping": c2p,
        "modes": modes,
        "conflicts": conflicts,
        "trace": trace,
        "status": "ok" if all(m in {"direct", "csp"} for m in modes) and not conflicts else "check",
    }

def trace_body_without_answer(trace):
    return trace.split("|A:", 1)[0]

def quarantine_unique_traces(rows, max_rows):
    """Reject duplicate trace bodies that point to conflicting answers."""
    body_to_answer = {}
    accepted, rejected = [], []
    for item in rows:
        body = trace_body_without_answer(item["trace"])
        bh = hashlib.sha256(body.encode("utf-8")).hexdigest()
        ans = item.get("answer", item.get("prediction", ""))
        if bh in body_to_answer and body_to_answer[bh] != ans:
            rejected.append({**item, "reject_reason": "duplicate_body_answer_conflict"})
            continue
        body_to_answer[bh] = ans
        accepted.append({**item, "body_hash": bh[:16]})
        if len(accepted) >= max_rows:
            break
    return accepted, rejected

if pd is None:
    REPAIR_TRACE_SUMMARY.update({"status": "skipped_no_pandas"})
else:
    train_csv = _find_csv_by_columns("train.csv")
    test_csv = _find_csv_by_columns("test.csv")
    if train_csv is None:
        print("[SubCipher] train.csv not found; skipping gate")
        REPAIR_TRACE_SUMMARY.update({"status": "skipped_no_train_csv"})
    else:
        train_df = pd.read_csv(train_csv)
        test_df = pd.read_csv(test_csv) if test_csv is not None else None
        vocab = build_vocab_from_prompts(train_df, test_df)
        sub_train = train_df[train_df["prompt"].map(is_substitution_prompt)].copy()
        print(f"[SubCipher] train_csv={train_csv}")
        print(f"[SubCipher] test_csv={test_csv}")
        print(f"[SubCipher] vocab_words={len(vocab)}")
        print(f"[SubCipher] train_rows={len(sub_train)}")

        solved_rows, failures = [], []
        for _, row in sub_train.iterrows():
            try:
                solved = solve_substitution_prompt(row["prompt"], vocab)
                ok = solved["prediction"] == str(row.get("answer", "")).strip().lower()
                item = {
                    "id": row["id"],
                    "category": "substitution_cipher",
                    "mode": "bijective_char_csp",
                    "prediction": solved["prediction"],
                    "answer": str(row.get("answer", "")).strip().lower(),
                    "ok": bool(ok),
                    "status": solved["status"],
                    "modes": " ".join(solved["modes"]),
                    "trace": solved["trace"],
                }
                solved_rows.append(item)
                if not ok:
                    failures.append(item)
            except Exception as e:
                failures.append({"id": row.get("id", "?"), "error": repr(e), "ok": False})

        val_path = REPAIR_DIR / "substitution_train_validation.csv"
        if solved_rows:
            pd.DataFrame(solved_rows).to_csv(val_path, index=False)
        correct = sum(1 for x in solved_rows if x.get("ok"))
        acc = correct / max(len(solved_rows), 1)
        print(f"[SubCipher] validation={correct}/{len(solved_rows)} acc={acc:.6f} failures={len(failures)}")

        trace_candidates = [x for x in solved_rows if x.get("ok")]
        accepted, rejected = quarantine_unique_traces(trace_candidates, SUB_REPAIR_MAX_ROWS)
        with REPAIR_TRACE_PATH.open("w", encoding="utf-8") as f:
            for item in accepted:
                f.write(json.dumps({
                    "id": item["id"],
                    "category": item["category"],
                    "mode": item["mode"],
                    "trace": item["trace"],
                    "answer": item["answer"],
                    "body_hash": item["body_hash"],
                }, ensure_ascii=False) + "\n")
        print(f"[SubCipher] compact_traces={len(accepted)} path={REPAIR_TRACE_PATH}")
        print(f"[SubCipher] rejected_conflicts={len(rejected)}")

        test_predictions = []
        if test_df is not None:
            for _, row in test_df.iterrows():
                if is_substitution_prompt(row["prompt"]):
                    solved = solve_substitution_prompt(row["prompt"], vocab)
                    test_predictions.append({
                        "id": row["id"],
                        "prediction": solved["prediction"],
                        "status": solved["status"],
                        "modes": " ".join(solved["modes"]),
                        "trace": solved["trace"],
                    })
            if test_predictions:
                test_pred_path = REPAIR_DIR / "substitution_visible_test_predictions.csv"
                pd.DataFrame(test_predictions).to_csv(test_pred_path, index=False)
                print("[SubCipher] visible_test_predictions:")
                for x in test_predictions[:10]:
                    print(f" - {x['id']}: {x['prediction']} [{x['status']}]")

        REPAIR_TRACE_SUMMARY.update({
            "status": "ok",
            "train_csv": str(train_csv),
            "test_csv": str(test_csv) if test_csv else None,
            "vocab_words": len(vocab),
            "train_rows": int(len(solved_rows)),
            "correct": int(correct),
            "accuracy": float(acc),
            "failures": int(len(failures)),
            "compact_traces": int(len(accepted)),
            "rejected_conflicts": int(len(rejected)),
            "trace_path": str(REPAIR_TRACE_PATH),
            "visible_test_predictions": int(len(test_predictions)),
        })

summary_path = REPAIR_DIR / "d3_repair_summary.json"
summary_path.write_text(json.dumps(REPAIR_TRACE_SUMMARY, indent=2), encoding="utf-8")
print("[SubCipher] summary:", json.dumps(REPAIR_TRACE_SUMMARY, sort_keys=True))


[SubCipher] train_csv=/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
[SubCipher] test_csv=/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv
[SubCipher] vocab_words=77
[SubCipher] train_rows=1576
[SubCipher] validation=1576/1576 acc=1.000000 failures=0
[SubCipher] compact_traces=200 path=/kaggle/working/d3_repair_traces/substitution_compact_csp_traces.jsonl
[SubCipher] rejected_conflicts=0
[SubCipher] visible_test_predictions:
 - 00189f6a: cat imagines book [ok]
[SubCipher] summary: {"accuracy": 1.0, "category": "substitution_cipher", "compact_traces": 200, "correct": 1576, "enabled": true, "failures": 0, "max_rows": 200, "rejected_conflicts": 0, "status": "ok", "test_csv": "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv", "trace_path": "/kaggle/working/d3_repair_traces/substitution_compact_csp_traces.jsonl", "train_csv": "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.

## 2B. D4 Cryptarithm Gate — solver-verified compact CoT generator

This cell parses the Alice equation-transformation cryptarithm rows, runs a bounded brute-force deducer, keeps only rows where the solver prediction exactly matches the training answer, and emits compact arithmetic traces.

The traces are written for audit / optional downstream LoRA data prep. They are intentionally **not** placed inside `submission.zip`; the evaluator expects only a LoRA adapter payload.


In [6]:

# -----------------------------------------------------------------------------
# D4 Solver-Verified Cryptarithm Gate
# -----------------------------------------------------------------------------
# Purpose:
# - Detect Alice equation-transformation cryptarithm rows.
# - Solve with bounded brute-force CSP over symbol->digit and op->operation.
# - Keep only solver predictions that exactly match train.csv answer.
# - Emit compact arithmetic CoT traces and optional dgxchen-compatible v2 CSV.
# - Do not include traces in submission.zip; competition expects a LoRA adapter zip.

from pathlib import Path
import os, re, json, time, signal
from collections import Counter

CRYPTO_MARKER = "secret set of transformation rules is applied to equations"
CRYPTO_REPAIR_DIR = Path("/kaggle/working/d4_repair_traces")
CRYPTO_REPAIR_DIR.mkdir(parents=True, exist_ok=True)
CRYPTO_REPAIR_MAX_ROWS = int(os.environ.get("CRYPTO_REPAIR_MAX_ROWS", "95"))
CRYPTO_SCAN_MAX_ROWS = int(os.environ.get("CRYPTO_SCAN_MAX_ROWS", "1555"))
CRYPTO_ROW_TIMEOUT_SEC = float(os.environ.get("CRYPTO_ROW_TIMEOUT_SEC", "0.75"))
CRYPTO_UPSAMPLE = int(os.environ.get("CRYPTO_UPSAMPLE", "12"))

CRYPTO_REPAIR_TRACE_PATH = CRYPTO_REPAIR_DIR / "cryptarithm_solver_verified_compact_traces.jsonl"
CRYPTO_V2_CSV_PATH = CRYPTO_REPAIR_DIR / "cryptarithm_solver_verified_v2.csv"

CRYPTO_TRACE_SUMMARY = {
    "enabled": True,
    "status": "not_run",
    "category": "cryptarithm",
    "max_verified_rows": CRYPTO_REPAIR_MAX_ROWS,
    "scan_max_rows": CRYPTO_SCAN_MAX_ROWS,
    "row_timeout_sec": CRYPTO_ROW_TIMEOUT_SEC,
}

_EQ_RE_CRYPTO = re.compile(r"^(\S{5})\s*=\s*(\S{1,4})\s*$")
_QUESTION_RE_CRYPTO = re.compile(r"Now,\s+determine\s+the\s+result\s+for:\s*(\S{5})\s*$")

CRYPTO_OPS = [
    lambda a, b: a + b,             # add
    lambda a, b: abs(a - b),        # abs_diff
    lambda a, b: a * b,             # mul
    lambda a, b: a * 100 + b,       # concat
    lambda a, b: b * 100 + a,       # rev_concat
]
CRYPTO_OP_NAMES = ["add", "abs_diff", "mul", "concat", "rev_concat"]

def is_cryptarithm_prompt(prompt):
    return isinstance(prompt, str) and CRYPTO_MARKER in prompt

def parse_cryptarithm_prompt(prompt):
    examples = []
    question = None
    for raw_line in str(prompt).splitlines():
        line = raw_line.strip()
        if not line:
            continue
        qm = _QUESTION_RE_CRYPTO.search(line)
        if qm:
            question = qm.group(1)
            continue
        em = _EQ_RE_CRYPTO.match(line)
        if em:
            examples.append({"input_value": em.group(1), "output_value": em.group(2)})
    if not examples or question is None:
        return None
    return {"examples": examples, "question": question}

def crypto_num_to_digits(n):
    if n == 0:
        return (0,)
    out = []
    while n > 0:
        out.append(n % 10)
        n //= 10
    return tuple(reversed(out))

class CryptarithmSolver:
    def __init__(self, examples, query, unique=True, strict_guess=True, max_solutions=200):
        self.examples = examples
        self.query = query
        self.unique = unique
        self.strict_guess = strict_guess
        self.max_solutions = max_solutions
        self.mapping = {}
        self.used = set()
        self.op_assign = {}
        self.answers = Counter()
        self.answer_info = {}
        self.guess_mode_answers = set()

    def solve(self):
        self._process(0)
        if not self.answers:
            return None, ({}, {})
        q_op = self.query[2] if len(self.query) == 5 else None
        example_ops = {ex[2] for ex in self.examples}
        is_guess = q_op is not None and q_op not in example_ops
        if is_guess and self.strict_guess and len(self.guess_mode_answers) > 1:
            return None, ({}, {})
        best, best_count = self.answers.most_common(1)[0]
        total = sum(self.answers.values())
        if not self.unique and total > 1 and best_count < total * 0.3:
            return None, ({}, {})
        return best, self.answer_info.get(best, ({}, {}))

    def _vals(self, sym):
        if sym in self.mapping:
            return (self.mapping[sym],)
        if self.unique:
            return tuple(d for d in range(10) if d not in self.used)
        return tuple(range(10))

    def _assign(self, sym, dig):
        if sym in self.mapping:
            return False if self.mapping[sym] == dig else None
        if self.unique and dig in self.used:
            return None
        self.mapping[sym] = dig
        if self.unique:
            self.used.add(dig)
        return True

    def _undo(self, sym, marker):
        if marker is True:
            dig = self.mapping.pop(sym)
            if self.unique:
                self.used.remove(dig)

    def _result_digits(self, op_id, left, right):
        val = CRYPTO_OPS[op_id](left, right)
        if op_id >= 3:
            if val < 0 or val >= 10000:
                return None
            return (val // 1000, (val // 100) % 10, (val // 10) % 10, val % 10)
        return crypto_num_to_digits(val)

    def _process(self, idx):
        if len(self.answers) >= self.max_solutions:
            return
        if idx == len(self.examples):
            self._compute_query()
            return

        s0, s1, op_sym, s3, s4, rsyms = self.examples[idx]
        rlen = len(rsyms)
        feasible_ops = []
        if rlen <= 3:
            feasible_ops.append(0)
        if rlen <= 2:
            feasible_ops.append(1)
        if rlen <= 4:
            feasible_ops.append(2)
        if rlen == 4:
            feasible_ops.extend([3, 4])

        for d0 in self._vals(s0):
            n0 = self._assign(s0, d0)
            if n0 is None:
                continue
            for d1 in self._vals(s1):
                n1 = self._assign(s1, d1)
                if n1 is None:
                    continue
                left = d0 * 10 + d1
                for d3 in self._vals(s3):
                    n3 = self._assign(s3, d3)
                    if n3 is None:
                        continue
                    for d4 in self._vals(s4):
                        n4 = self._assign(s4, d4)
                        if n4 is None:
                            continue
                        right = d3 * 10 + d4
                        ops_to_try = [self.op_assign[op_sym]] if op_sym in self.op_assign else feasible_ops
                        for op_id in ops_to_try:
                            rd = self._result_digits(op_id, left, right)
                            if rd is None or len(rd) != rlen:
                                continue
                            assigned = []
                            ok = True
                            for rsym, rdig in zip(rsyms, rd):
                                marker = self._assign(rsym, rdig)
                                if marker is None:
                                    ok = False
                                    break
                                assigned.append((rsym, marker))
                            if ok:
                                op_new = op_sym not in self.op_assign
                                if op_new:
                                    self.op_assign[op_sym] = op_id
                                self._process(idx + 1)
                                if op_new:
                                    del self.op_assign[op_sym]
                            for rsym, marker in reversed(assigned):
                                self._undo(rsym, marker)
                            if len(self.answers) >= self.max_solutions:
                                self._undo(s4, n4); self._undo(s3, n3); self._undo(s1, n1); self._undo(s0, n0)
                                return
                        self._undo(s4, n4)
                    self._undo(s3, n3)
                self._undo(s1, n1)
            self._undo(s0, n0)

    def _compute_query(self):
        if len(self.query) != 5:
            return
        s0, s1, op_sym, s3, s4 = tuple(self.query)
        op_new = op_sym not in self.op_assign
        for d0 in self._vals(s0):
            n0 = self._assign(s0, d0)
            if n0 is None:
                continue
            for d1 in self._vals(s1):
                n1 = self._assign(s1, d1)
                if n1 is None:
                    continue
                left = d0 * 10 + d1
                for d3 in self._vals(s3):
                    n3 = self._assign(s3, d3)
                    if n3 is None:
                        continue
                    for d4 in self._vals(s4):
                        n4 = self._assign(s4, d4)
                        if n4 is None:
                            continue
                        right = d3 * 10 + d4
                        ops_to_try = [self.op_assign[op_sym]] if op_sym in self.op_assign else list(range(5))
                        for op_id in ops_to_try:
                            rd = self._result_digits(op_id, left, right)
                            if rd is None:
                                continue
                            inv = {v: k for k, v in self.mapping.items()}
                            if any(d not in inv for d in rd):
                                continue
                            ans = "".join(inv[d] for d in rd)
                            self.answers[ans] += 1
                            if op_new:
                                self.guess_mode_answers.add(ans)
                            if ans not in self.answer_info:
                                m = dict(self.mapping)
                                ops = dict(self.op_assign)
                                if op_new:
                                    ops[op_sym] = op_id
                                self.answer_info[ans] = (m, ops)
                        self._undo(s4, n4)
                    self._undo(s3, n3)
                self._undo(s1, n1)
            self._undo(s0, n0)

def solve_cryptarithm_problem(parsed):
    examples = []
    for ex in parsed["examples"]:
        inp = ex["input_value"]
        out = ex["output_value"]
        if len(inp) != 5 or not (1 <= len(out) <= 4):
            return None, ({}, {})
        examples.append((inp[0], inp[1], inp[2], inp[3], inp[4], tuple(out)))
    query = parsed["question"]
    if len(query) != 5:
        return None, ({}, {})
    return CryptarithmSolver(examples, query).solve()

class _CryptoTimeout(Exception):
    pass

def _crypto_alarm_handler(signum, frame):
    raise _CryptoTimeout()

def solve_cryptarithm_with_timeout(parsed, seconds=0.75):
    if hasattr(signal, "SIGALRM"):
        old_handler = signal.signal(signal.SIGALRM, _crypto_alarm_handler)
        signal.setitimer(signal.ITIMER_REAL, max(float(seconds), 0.05))
        try:
            return solve_cryptarithm_problem(parsed), "ok"
        except _CryptoTimeout:
            return (None, ({}, {})), "timeout"
        except Exception as e:
            return (None, ({}, {})), f"error:{type(e).__name__}"
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old_handler)
    try:
        return solve_cryptarithm_problem(parsed), "ok"
    except Exception as e:
        return (None, ({}, {})), f"error:{type(e).__name__}"

def decode_crypto_input(inp, mapping):
    if len(inp) != 5:
        return "??", "?", "??"
    a = mapping.get(inp[0], "?") * 10 + mapping.get(inp[1], 0) if inp[0] in mapping and inp[1] in mapping else "??"
    b = mapping.get(inp[3], "?") * 10 + mapping.get(inp[4], 0) if inp[3] in mapping and inp[4] in mapping else "??"
    return a, inp[2], b

def make_cryptarithm_trace(row_id, parsed, prediction, mapping, op_assign):
    map_s = ",".join(f"{k}={v}" for k, v in sorted(mapping.items(), key=lambda kv: str(kv[0])))
    op_s = ",".join(f"{k}={CRYPTO_OP_NAMES[v]}" for k, v in sorted(op_assign.items(), key=lambda kv: str(kv[0])))
    parts = [f"CRYPTO|id={row_id}", f"M:{map_s}", f"OPS:{op_s}"]
    for i, ex in enumerate(parsed["examples"][:4]):
        inp, out = ex["input_value"], ex["output_value"]
        left, op_sym, right = decode_crypto_input(inp, mapping)
        op_name = CRYPTO_OP_NAMES[op_assign[op_sym]] if op_sym in op_assign else "unk"
        parts.append(f"E{i}:{inp}->{left}:{op_name}:{right}={out}")
    q = parsed["question"]
    qleft, qop, qright = decode_crypto_input(q, mapping)
    qop_name = CRYPTO_OP_NAMES[op_assign[qop]] if qop in op_assign else "guess"
    parts.append(f"Q:{q}->{qleft}:{qop_name}:{qright}")
    parts.append(f"A:{prediction}")
    return "|".join(parts)

def _find_optional_dgx_csv():
    candidates = [
        Path("/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"),
        Path("/kaggle/input/nemotron-cot-tong/problem_ids_matched.csv"),
        Path("/kaggle/input/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            hits = list(root.rglob("problem_ids_matched.csv"))
            if hits:
                return hits[0]
    return None

def run_cryptarithm_gate():
    global CRYPTO_TRACE_SUMMARY
    if pd is None:
        print("[CryptoGate] pandas unavailable; skipping")
        CRYPTO_TRACE_SUMMARY.update({"status": "skipped_no_pandas"})
        return CRYPTO_TRACE_SUMMARY

    train_path = _find_csv_by_columns("train.csv")
    test_path = _find_csv_by_columns("test.csv")
    if train_path is None:
        print("[CryptoGate] train.csv not found; skipping")
        CRYPTO_TRACE_SUMMARY.update({"status": "skipped_no_train"})
        return CRYPTO_TRACE_SUMMARY

    train_df = pd.read_csv(train_path)
    crypto_df = train_df[train_df["prompt"].map(is_cryptarithm_prompt)].copy()
    total_crypto = int(len(crypto_df))
    print(f"[CryptoGate] train_csv={train_path}")
    print(f"[CryptoGate] rows={total_crypto}")
    print(f"[CryptoGate] target_verified={CRYPTO_REPAIR_MAX_ROWS} timeout={CRYPTO_ROW_TIMEOUT_SEC}s")

    verified = []
    stats = Counter()
    t0 = time.time()
    for scanned, (_idx, row) in enumerate(crypto_df.head(CRYPTO_SCAN_MAX_ROWS).iterrows(), start=1):
        parsed = parse_cryptarithm_prompt(row["prompt"])
        if parsed is None:
            stats["parse_fail"] += 1
            continue
        (pred, info), status = solve_cryptarithm_with_timeout(parsed, CRYPTO_ROW_TIMEOUT_SEC)
        stats[status] += 1
        if pred is None:
            stats["abstain"] += 1
            continue
        if str(pred) == str(row["answer"]):
            mapping, ops = info
            trace = make_cryptarithm_trace(row["id"], parsed, pred, mapping, ops)
            h = hashlib.sha256((str(row["prompt"]) + "\n" + trace + "\n" + str(row["answer"])).encode("utf-8", errors="replace")).hexdigest()
            verified.append({
                "id": row["id"],
                "prompt": row["prompt"],
                "answer": row["answer"],
                "category": "cryptarithm",
                "mode": "solver_verified_compact_arithmetic_cot",
                "generated_cot": trace,
                "trace_sha256": h,
            })
            stats["verified_exact"] += 1
        else:
            stats["wrong_prediction"] += 1
        if len(verified) >= CRYPTO_REPAIR_MAX_ROWS:
            break
        if scanned % 100 == 0:
            print(f"[CryptoGate] scanned={scanned} verified={len(verified)} abstain={stats['abstain']} timeout={stats['timeout']}")

    # Duplicate / conflict quarantine.
    trace_to_answer = {}
    clean = []
    conflicts = []
    for item in verified:
        body = item["generated_cot"]
        ans = str(item["answer"])
        if body in trace_to_answer and trace_to_answer[body] != ans:
            conflicts.append({"id": item["id"], "old_answer": trace_to_answer[body], "new_answer": ans})
            continue
        trace_to_answer[body] = ans
        clean.append(item)

    with CRYPTO_REPAIR_TRACE_PATH.open("w", encoding="utf-8") as f:
        for item in clean:
            f.write(json.dumps({
                "id": item["id"],
                "category": item["category"],
                "mode": item["mode"],
                "generated_cot": item["generated_cot"],
                "answer": item["answer"],
                "trace_sha256": item["trace_sha256"],
            }, ensure_ascii=False) + "\n")

    v2_rows = []
    for item in clean:
        base = {
            "id": item["id"],
            "prompt": item["prompt"],
            "answer": item["answer"],
            "type": "cryptarithm_solver_verified",
            "generated_cot": item["generated_cot"],
        }
        for _ in range(CRYPTO_UPSAMPLE):
            v2_rows.append(dict(base))
    if v2_rows:
        pd.DataFrame(v2_rows).to_csv(CRYPTO_V2_CSV_PATH, index=False)

    dgx_path = _find_optional_dgx_csv()
    dgx_v2_path = None
    if dgx_path is not None and v2_rows:
        try:
            dgx = pd.read_csv(dgx_path)
            verified_ids = {str(x["id"]) for x in clean}
            base = dgx[~dgx["id"].astype(str).isin(verified_ids)].copy()
            dgx_v2 = pd.concat([base, pd.DataFrame(v2_rows)], ignore_index=True)
            dgx_v2_path = CRYPTO_REPAIR_DIR / "problem_ids_matched_crypto_v2_solver_verified.csv"
            dgx_v2.to_csv(dgx_v2_path, index=False)
            print(f"[CryptoGate] optional_dgx_v2={dgx_v2_path} rows={len(dgx_v2)}")
        except Exception as e:
            print("[CryptoGate] optional dgx v2 build failed:", repr(e))

    elapsed = time.time() - t0
    summary = {
        "enabled": True,
        "status": "complete",
        "category": "cryptarithm",
        "train_csv": str(train_path),
        "test_csv": str(test_path) if test_path else None,
        "train_rows": int(total_crypto),
        "scanned_rows": int(min(scanned if total_crypto else 0, CRYPTO_SCAN_MAX_ROWS)),
        "verified_exact": int(len(clean)),
        "raw_verified": int(len(verified)),
        "rejected_conflicts": int(len(conflicts)),
        "trace_path": str(CRYPTO_REPAIR_TRACE_PATH),
        "v2_csv_path": str(CRYPTO_V2_CSV_PATH) if v2_rows else None,
        "dgx_v2_csv_path": str(dgx_v2_path) if dgx_v2_path else None,
        "upsample": CRYPTO_UPSAMPLE,
        "stats": dict(stats),
        "elapsed_sec": round(elapsed, 3),
    }
    CRYPTO_TRACE_SUMMARY = summary
    print("[CryptoGate] summary:", json.dumps(summary, sort_keys=True))
    if clean:
        print("[CryptoGate] first_trace:", clean[0]["generated_cot"][:500])
    return summary

try:
    CRYPTO_TRACE_SUMMARY = run_cryptarithm_gate()
except Exception as e:
    CRYPTO_TRACE_SUMMARY = {
        "enabled": True,
        "status": "error",
        "error": repr(e),
        "category": "cryptarithm",
    }
    print("[CryptoGate] ERROR:", repr(e))


[CryptoGate] train_csv=/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
[CryptoGate] rows=1555
[CryptoGate] target_verified=95 timeout=0.75s
[CryptoGate] scanned=800 verified=47 abstain=707 timeout=38
[CryptoGate] summary: {"category": "cryptarithm", "dgx_v2_csv_path": null, "elapsed_sec": 119.696, "enabled": true, "raw_verified": 95, "rejected_conflicts": 0, "scanned_rows": 1505, "stats": {"abstain": 1321, "ok": 1429, "timeout": 76, "verified_exact": 95, "wrong_prediction": 89}, "status": "complete", "test_csv": "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv", "trace_path": "/kaggle/working/d4_repair_traces/cryptarithm_solver_verified_compact_traces.jsonl", "train_csv": "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv", "train_rows": 1555, "upsample": 12, "v2_csv_path": "/kaggle/working/d4_repair_traces/cryptarithm_solver_verified_v2.csv", "verified_exact": 95}
[CryptoGate] first_trace: CRYPTO

## 3. Apply GlyphMatics v8 PairFold fused-projection transport patch

This patch intercepts Tinker fused projection merging and compresses oversized merged LoRA pairs to the required fused rank.

### What is fully defined here

| Area | Implementation |
|---|---|
| Fused merge | Builds one fused LoRA pair from separate projection components |
| Dense delta | Computes `Delta = B @ A` in fp32 |
| Rank compression | Uses SVD and keeps `FORCED_FUSED_RANK` directions |
| Residual matching | Builds block-energy signatures for kept and discarded directions |
| Pair compression | Groups kept ranks into pairs and matches tail residuals to closest pair |
| Gain transport | Redistributes safe restoration gain toward matched pairs |
| Optional tail orientation | Tiny capped vector blend into matched survivor directions |
| Balanced factorization | Splits scale symmetrically across LoRA `A` and `B` |
| RowGuard | Clips row-level overshoot after reconstruction |
| Ledger | Writes a transport ledger into the output adapter folder |

The defaults are intentionally conservative. They are designed to change *where* restoration is placed, not to brute-force more global energy.


In [7]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import os
import hashlib
import math

import torch
import tinker_cookbook.weights._adapter as A

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# -----------------------------------------------------------------------------
# Competition knobs
# -----------------------------------------------------------------------------
FORCED_FUSED_RANK = int(os.environ.get("FORCED_FUSED_RANK", "32"))
SVD_ENERGY_GAIN_CAP = float(os.environ.get("SVD_ENERGY_GAIN_CAP", "1.19"))
GAIN_DISTRIBUTION = "pairfold_residual_matched_deterministic_rebuild_rowguard"

DETERMINISTIC_REBUILD_ENABLED = os.environ.get("DETERMINISTIC_REBUILD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
DETERMINISTIC_REBUILD_ACCEPT_EPS = float(os.environ.get("DETERMINISTIC_REBUILD_ACCEPT_EPS", "0.0000"))

ROW_NORM_GUARD_ENABLED = os.environ.get("ROW_NORM_GUARD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
ROW_NORM_GAIN_CAP = float(os.environ.get("ROW_NORM_GAIN_CAP", "1.08"))

PAIRFOLD_ENABLED = os.environ.get("PAIRFOLD_ENABLED", "1").strip().lower() not in {"0", "false", "no", "off"}
PAIRFOLD_PAIR_WIDTH = int(os.environ.get("PAIRFOLD_PAIR_WIDTH", "2"))
PAIRFOLD_TAIL_MAX = int(os.environ.get("PAIRFOLD_TAIL_MAX", "96"))
PAIRFOLD_MIN_SIM = float(os.environ.get("PAIRFOLD_MIN_SIM", "0.70"))
PAIRFOLD_MAX_GAIN_DELTA = float(os.environ.get("PAIRFOLD_MAX_GAIN_DELTA", "0.022"))
PAIRFOLD_VECTOR_BLEND = float(os.environ.get("PAIRFOLD_VECTOR_BLEND", "0.025"))
PAIRFOLD_DECAY = float(os.environ.get("PAIRFOLD_DECAY", "0.985"))
PAIRFOLD_COL_BINS = int(os.environ.get("PAIRFOLD_COL_BINS", "8"))

DUAL_PAIR_ENABLED = os.environ.get("DUAL_PAIR_ENABLED", "0").strip().lower() not in {"0", "false", "no", "off"}
DUAL_PAIR_SPLIT_RAW = os.environ.get("DUAL_PAIR_SPLIT", "0.62,0.58")

if FORCED_FUSED_RANK <= 0:
    raise ValueError("FORCED_FUSED_RANK must be positive")
if not (1.0 <= SVD_ENERGY_GAIN_CAP <= 1.25):
    raise ValueError("SVD_ENERGY_GAIN_CAP must stay inside [1.0, 1.25]")
if not (1.0 <= ROW_NORM_GAIN_CAP <= 1.25):
    raise ValueError("ROW_NORM_GAIN_CAP must stay inside [1.0, 1.25]")
if PAIRFOLD_PAIR_WIDTH <= 0:
    raise ValueError("PAIRFOLD_PAIR_WIDTH must be positive")
if PAIRFOLD_TAIL_MAX < 0:
    raise ValueError("PAIRFOLD_TAIL_MAX must be non-negative")
if not (0.0 <= PAIRFOLD_MIN_SIM <= 1.0):
    raise ValueError("PAIRFOLD_MIN_SIM must be inside [0, 1]")
if not (0.0 <= PAIRFOLD_MAX_GAIN_DELTA <= 0.10):
    raise ValueError("PAIRFOLD_MAX_GAIN_DELTA must be inside [0, 0.10]")
if not (0.0 <= PAIRFOLD_VECTOR_BLEND <= 0.20):
    raise ValueError("PAIRFOLD_VECTOR_BLEND must be inside [0, 0.20]")
if not (0.50 <= PAIRFOLD_DECAY <= 1.0):
    raise ValueError("PAIRFOLD_DECAY must be inside [0.50, 1.0]")
if not (1 <= PAIRFOLD_COL_BINS <= 64):
    raise ValueError("PAIRFOLD_COL_BINS must be inside [1, 64]")
if not (-0.05 <= DETERMINISTIC_REBUILD_ACCEPT_EPS <= 0.05):
    raise ValueError("DETERMINISTIC_REBUILD_ACCEPT_EPS must be inside [-0.05, 0.05]")

def _parse_pair_split(raw: str):
    vals = []
    for piece in raw.replace(";", ",").split(","):
        piece = piece.strip()
        if piece:
            vals.append(float(piece))
    if len(vals) != 2:
        raise ValueError("DUAL_PAIR_SPLIT must contain exactly two numbers, e.g. '0.62,0.58'")
    if not all(v > 0 for v in vals):
        raise ValueError("DUAL_PAIR_SPLIT values must be positive")
    return vals

DUAL_PAIR_SPLIT = _parse_pair_split(DUAL_PAIR_SPLIT_RAW)

# -----------------------------------------------------------------------------
# Ledger
# -----------------------------------------------------------------------------
class GlyphmaticTransportLedger:
    def __init__(self):
        self.events = []
        self.counts = Counter()

    def emit(self, *, src, dst, op, alpha, beta=None, gamma=None):
        beta = beta or {}
        gamma = gamma or {}
        basis = json.dumps(
            {"alpha": alpha, "src": str(src), "dst": str(dst), "op": str(op), "beta": beta, "gamma": gamma},
            sort_keys=True,
            default=str,
        )
        event = {
            "alpha": str(alpha),
            "source": str(src),
            "dest": str(dst),
            "op": str(op),
            "beta": beta,
            "gamma": gamma,
            "verification_hash": hashlib.sha256(basis.encode("utf-8")).hexdigest()[:16],
        }
        self.events.append(event)
        self.counts[(event["alpha"], event["op"])] += 1

    def markdown(self) -> str:
        lines = [
            "# GlyphMatics Transport Ledger",
            "",
            "Generated during tinker-cookbook adapter conversion.",
            "",
            "## Submission configuration",
            "",
            f"- `FORCED_FUSED_RANK`: `{FORCED_FUSED_RANK}`",
            f"- `SVD_ENERGY_GAIN_CAP`: `{SVD_ENERGY_GAIN_CAP}`",
            f"- `GAIN_DISTRIBUTION`: `{GAIN_DISTRIBUTION}`",
            f"- `DETERMINISTIC_REBUILD_ENABLED`: `{DETERMINISTIC_REBUILD_ENABLED}`",
            f"- `DETERMINISTIC_REBUILD_ACCEPT_EPS`: `{DETERMINISTIC_REBUILD_ACCEPT_EPS}`",
            f"- `PAIRFOLD_ENABLED`: `{PAIRFOLD_ENABLED}`",
            f"- `PAIRFOLD_PAIR_WIDTH`: `{PAIRFOLD_PAIR_WIDTH}`",
            f"- `PAIRFOLD_TAIL_MAX`: `{PAIRFOLD_TAIL_MAX}`",
            f"- `PAIRFOLD_MIN_SIM`: `{PAIRFOLD_MIN_SIM}`",
            f"- `PAIRFOLD_MAX_GAIN_DELTA`: `{PAIRFOLD_MAX_GAIN_DELTA}`",
            f"- `PAIRFOLD_VECTOR_BLEND`: `{PAIRFOLD_VECTOR_BLEND}`",
            f"- `PAIRFOLD_DECAY`: `{PAIRFOLD_DECAY}`",
            f"- `PAIRFOLD_COL_BINS`: `{PAIRFOLD_COL_BINS}`",
            f"- `ROW_NORM_GUARD_ENABLED`: `{ROW_NORM_GUARD_ENABLED}`",
            f"- `ROW_NORM_GAIN_CAP`: `{ROW_NORM_GAIN_CAP}`",
            f"- `DUAL_PAIR_ENABLED`: `{DUAL_PAIR_ENABLED}`",
            f"- `DUAL_PAIR_SPLIT`: `{DUAL_PAIR_SPLIT}`",
            "",
            "## Event summary",
            "",
            "| alpha | op | count |",
            "|---|---|---:|",
        ]
        for (alpha, op), count in sorted(self.counts.items()):
            lines.append(f"| `{alpha}` | `{op}` | {count} |")

        lines += [
            "",
            "## First 80 events",
            "",
            "| # | alpha | op | source | destination | gamma | hash |",
            "|---:|---|---|---|---|---|---|",
        ]
        for i, event in enumerate(self.events[:80], 1):
            gamma = json.dumps(event["gamma"], sort_keys=True, default=str)
            lines.append(
                f"| {i} | `{event['alpha']}` | `{event['op']}` | "
                f"`{event['source']}` | `{event['dest']}` | `{gamma}` | `{event['verification_hash']}` |"
            )
        return "\n".join(lines) + "\n"

    def print_summary(self):
        print("[GlyphMatics ledger] events:", len(self.events))
        for (alpha, op), count in sorted(self.counts.items()):
            print(f"[GlyphMatics ledger] {alpha}:{op}={count}")

GLYPH_LEDGER = GlyphmaticTransportLedger()

# -----------------------------------------------------------------------------
# Numeric helpers
# -----------------------------------------------------------------------------
def _safe_unit(x: torch.Tensor, dim=None, eps: float = 1e-12):
    if dim is None:
        return x / torch.linalg.vector_norm(x).clamp_min(eps)
    return x / torch.linalg.vector_norm(x, dim=dim, keepdim=True).clamp_min(eps)

def _make_even_blocks(length: int, count: int, prefix: str):
    count = max(1, min(int(count), int(length)))
    blocks = []
    for i in range(count):
        start = int(round(i * length / count))
        end = int(round((i + 1) * length / count))
        if end > start:
            blocks.append((start, end, f"{prefix}{i}"))
    return blocks

def _make_row_blocks(row_count: int, component_slices=None):
    blocks = []
    if component_slices:
        for row_start, row_end, _rank, name in component_slices:
            row_start = max(0, min(int(row_start), int(row_count)))
            row_end = max(row_start, min(int(row_end), int(row_count)))
            if row_end > row_start:
                blocks.append((row_start, row_end, str(name)))
    covered = sum(e - s for s, e, _ in blocks)
    if not blocks or covered < row_count:
        # Fallback also covers non-component rows in unusual fused layouts.
        blocks = _make_even_blocks(row_count, min(8, row_count), "rowbin")
    return blocks

def _block_energy(vec: torch.Tensor, blocks):
    vals = []
    vec = vec.float()
    sq = vec * vec
    for start, end, _name in blocks:
        vals.append(sq[start:end].sum())
    out = torch.stack(vals) if vals else torch.ones(1, device=vec.device, dtype=torch.float32)
    return _safe_unit(out.float())

def _direction_signature(u_col: torch.Tensor, vh_row: torch.Tensor, row_blocks, col_bins: int):
    """
    Local behavior signature for residual matching.

    Raw SVD vectors are orthogonal globally, so direct cosine is not useful.
    This signature compares where a direction spends energy by fused output rows
    and input-column bins. Tail directions are folded into survivor pairs with
    similar local behavior.
    """
    col_blocks = _make_even_blocks(int(vh_row.numel()), max(1, min(col_bins, int(vh_row.numel()))), "colbin")
    row_sig = _block_energy(u_col, row_blocks)
    col_sig = _block_energy(vh_row, col_blocks)
    u_abs = u_col.float().abs()
    v_abs = vh_row.float().abs()
    moments = torch.tensor(
        [
            float(u_abs.max().detach().cpu()),
            float(v_abs.max().detach().cpu()),
            float(u_abs.mean().detach().cpu()),
            float(v_abs.mean().detach().cpu()),
        ],
        device=u_col.device,
        dtype=torch.float32,
    )
    return _safe_unit(torch.cat([row_sig, col_sig, _safe_unit(moments)]).float())

def _rank_pair_groups(rank: int, pair_width: int):
    groups = []
    start = 0
    while start < rank:
        end = min(rank, start + pair_width)
        groups.append(list(range(start, end)))
        start = end
    return groups

def _dual_lane_gain_vector(*, singular_values: torch.Tensor, raw_gain: torch.Tensor, rank: int):
    """
    Stable two-lane option retained for A/B tests.

    By default DUAL_PAIR_ENABLED=0 because PairFold is now the primary transport.
    """
    base_gain = torch.clamp(raw_gain, min=1.0, max=SVD_ENERGY_GAIN_CAP)
    gain_vec = torch.full_like(singular_values, fill_value=float(base_gain))

    if not DUAL_PAIR_ENABLED or rank < 2:
        return gain_vec, {
            "mode": "single_lane_before_pairfold",
            "raw_energy_gain": float(raw_gain.detach().cpu()),
            "global_energy_gain": float(base_gain.detach().cpu()),
            "lane_ranks": [int(rank)],
            "lane_caps": [float(SVD_ENERGY_GAIN_CAP)],
            "lane_gains": [float(base_gain.detach().cpu())],
        }

    first = rank // 2
    second = rank - first
    pair_mean = sum(DUAL_PAIR_SPLIT) / 2.0
    cap_excess = max(SVD_ENERGY_GAIN_CAP - 1.0, 0.0)

    lane_caps = [
        min(SVD_ENERGY_GAIN_CAP, 1.0 + cap_excess * (DUAL_PAIR_SPLIT[0] / pair_mean)),
        min(SVD_ENERGY_GAIN_CAP, 1.0 + cap_excess * (DUAL_PAIR_SPLIT[1] / pair_mean)),
    ]

    lane_gain_0 = torch.clamp(raw_gain, min=1.0, max=lane_caps[0])
    lane_gain_1 = torch.clamp(raw_gain, min=1.0, max=lane_caps[1])

    gain_vec[:first] = lane_gain_0
    gain_vec[first:first + second] = lane_gain_1

    return gain_vec, {
        "mode": "dual_pair_before_pairfold",
        "raw_energy_gain": float(raw_gain.detach().cpu()),
        "global_energy_gain_cap": float(SVD_ENERGY_GAIN_CAP),
        "dual_pair_split": [float(x) for x in DUAL_PAIR_SPLIT],
        "lane_ranks": [int(first), int(second)],
        "lane_caps": [float(x) for x in lane_caps],
        "lane_gains": [float(lane_gain_0.detach().cpu()), float(lane_gain_1.detach().cpu())],
    }

def _pairfold_transport(U: torch.Tensor, S: torch.Tensor, Vh: torch.Tensor, rank: int, gain_vec: torch.Tensor, row_blocks):
    """
    Residual-matched rank-pair compression.

    The discarded SVD tail is not directly retained as extra rank. Instead:
    1. Each kept direction gets a local behavior signature.
    2. Adjacent kept directions are grouped into rank pairs.
    3. Tail directions are matched to the closest surviving pair by signature.
    4. Restoration gain is redistributed toward pairs that absorbed similar tail.
    5. A tiny capped orientation blend can nudge the survivor pair toward the tail.

    Global gain is energy-renormalized back near the incoming target so this
    changes transport shape more than total volume.
    """
    device = U.device
    rank = int(rank)
    tail_available = max(0, int(S.numel()) - rank)
    if (not PAIRFOLD_ENABLED) or rank <= 0 or tail_available <= 0 or PAIRFOLD_TAIL_MAX <= 0:
        return U[:, :rank].contiguous(), Vh[:rank, :].contiguous(), gain_vec.contiguous(), {
            "pairfold_enabled": bool(PAIRFOLD_ENABLED),
            "pairfold_mode": "disabled_or_no_tail",
            "pairfold_assignments": 0,
            "pairfold_tail_used": 0,
        }

    U_k = U[:, :rank].clone()
    Vh_k = Vh[:rank, :].clone()
    S_k = S[:rank]
    gain_in = gain_vec.clone()

    head_sigs = torch.stack([
        _direction_signature(U[:, i], Vh[i, :], row_blocks, PAIRFOLD_COL_BINS)
        for i in range(rank)
    ])

    groups = _rank_pair_groups(rank, PAIRFOLD_PAIR_WIDTH)
    group_sigs = []
    for group in groups:
        idx = torch.tensor(group, device=device, dtype=torch.long)
        weights = S_k[idx].float().clamp_min(1e-12)
        weights = weights / weights.sum().clamp_min(1e-12)
        sig = (head_sigs[idx] * weights.unsqueeze(1)).sum(dim=0)
        group_sigs.append(_safe_unit(sig))
    group_sigs = torch.stack(group_sigs)

    pair_priority = torch.zeros(len(groups), device=device, dtype=torch.float32)
    head_priority = torch.zeros(rank, device=device, dtype=torch.float32)
    u_acc = torch.zeros_like(U_k.float())
    v_acc = torch.zeros_like(Vh_k.float())

    tail_count = min(tail_available, int(PAIRFOLD_TAIL_MAX))
    assignments = 0
    sim_sum = 0.0
    max_sim = 0.0

    for local_j in range(tail_count):
        j = rank + local_j
        tail_sig = _direction_signature(U[:, j], Vh[j, :], row_blocks, PAIRFOLD_COL_BINS)
        sims = group_sigs @ tail_sig
        best_group = int(torch.argmax(sims).detach().cpu())
        best_sim = float(sims[best_group].detach().cpu())
        if best_sim < PAIRFOLD_MIN_SIM:
            continue

        decay = float(PAIRFOLD_DECAY ** local_j)
        # Priority is relative, not raw energy injection. It tells where the
        # already-safe restoration should be placed.
        rel_tail = float((S[j] / S_k.mean().clamp_min(1e-12)).detach().cpu())
        priority = max(0.0, best_sim * decay * rel_tail)
        if priority <= 0:
            continue

        assignments += 1
        sim_sum += best_sim
        max_sim = max(max_sim, best_sim)
        pair_priority[best_group] += priority

        group = groups[best_group]
        idx = torch.tensor(group, device=device, dtype=torch.long)
        local_sims = (head_sigs[idx] @ tail_sig).clamp_min(0.0)
        local_weights = local_sims + S_k[idx].float() / S_k[idx].float().sum().clamp_min(1e-12)
        local_weights = local_weights / local_weights.sum().clamp_min(1e-12)

        for pos, head_idx in enumerate(group):
            head_priority[head_idx] += priority * float(local_weights[pos].detach().cpu())
            if PAIRFOLD_VECTOR_BLEND > 0:
                # Tiny orientation fold. Capped per tail and scaled by similarity.
                blend = min(
                    PAIRFOLD_VECTOR_BLEND,
                    PAIRFOLD_VECTOR_BLEND * best_sim * rel_tail,
                ) * float(local_weights[pos].detach().cpu())
                u_acc[:, head_idx] += float(blend) * U[:, j].float()
                v_acc[head_idx, :] += float(blend) * Vh[j, :].float()

    if assignments == 0 or float(head_priority.sum().detach().cpu()) <= 0.0:
        return U_k.contiguous(), Vh_k.contiguous(), gain_in.contiguous(), {
            "pairfold_enabled": True,
            "pairfold_mode": "no_similarity_match",
            "pairfold_assignments": 0,
            "pairfold_tail_used": int(tail_count),
            "pairfold_min_sim": float(PAIRFOLD_MIN_SIM),
        }

    # Rank-pair gain redistribution.
    # Center around 1.0, cap the delta, then energy-renormalize back to the
    # pre-PairFold target. This avoids simply making the adapter louder.
    priority = head_priority / head_priority.mean().clamp_min(1e-12)
    delta = (priority - 1.0).clamp(-1.0, 1.0) * float(PAIRFOLD_MAX_GAIN_DELTA)
    gain_out = gain_in * (1.0 + delta)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    target_energy = torch.sqrt(torch.sum((S_k * gain_in) ** 2)).clamp_min(1e-12)
    current_energy = torch.sqrt(torch.sum((S_k * gain_out) ** 2)).clamp_min(1e-12)
    gain_out = gain_out * (target_energy / current_energy)
    gain_out = torch.clamp(gain_out, min=1.0, max=float(SVD_ENERGY_GAIN_CAP))

    if PAIRFOLD_VECTOR_BLEND > 0:
        U_k = _safe_unit(U_k.float() + u_acc, dim=0).to(U.dtype)
        Vh_k = _safe_unit(Vh_k.float() + v_acc, dim=1).to(Vh.dtype)

    return U_k.contiguous(), Vh_k.contiguous(), gain_out.contiguous(), {
        "pairfold_enabled": True,
        "pairfold_mode": "residual_matched_rank_pairs",
        "pairfold_pair_width": int(PAIRFOLD_PAIR_WIDTH),
        "pairfold_tail_used": int(tail_count),
        "pairfold_assignments": int(assignments),
        "pairfold_assignment_rate": float(assignments / max(1, tail_count)),
        "pairfold_avg_match_sim": float(sim_sum / max(1, assignments)),
        "pairfold_max_match_sim": float(max_sim),
        "pairfold_min_sim": float(PAIRFOLD_MIN_SIM),
        "pairfold_max_gain_delta": float(PAIRFOLD_MAX_GAIN_DELTA),
        "pairfold_vector_blend": float(PAIRFOLD_VECTOR_BLEND),
        "pairfold_gain_mean": float(gain_out.mean().detach().cpu()),
        "pairfold_gain_min": float(gain_out.min().detach().cpu()),
        "pairfold_gain_max": float(gain_out.max().detach().cpu()),
    }

def _apply_row_norm_guard(B_new: torch.Tensor, A_new: torch.Tensor, delta: torch.Tensor):
    """Clip local row-level overshoot after compression."""
    if not ROW_NORM_GUARD_ENABLED:
        return B_new.contiguous(), {
            "row_norm_guard_enabled": False,
            "row_norm_gain_cap": float(ROW_NORM_GAIN_CAP),
            "row_clip_fraction": 0.0,
        }

    with torch.no_grad():
        recon = B_new.float() @ A_new.float()
        orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
        new_row = torch.linalg.vector_norm(recon, ord=2, dim=1).clamp_min(1e-12)
        ratio = new_row / orig_row
        max_allowed = orig_row * float(ROW_NORM_GAIN_CAP)
        row_scale = torch.minimum(torch.ones_like(new_row), max_allowed / new_row)
        clipped = row_scale < 0.999
        B_guarded = (B_new.float() * row_scale.unsqueeze(1)).to(B_new.dtype).contiguous()

    return B_guarded, {
        "row_norm_guard_enabled": True,
        "row_norm_gain_cap": float(ROW_NORM_GAIN_CAP),
        "row_clip_fraction": float(clipped.float().mean().detach().cpu()),
        "row_clip_count": int(clipped.sum().detach().cpu()),
        "row_ratio_mean_before_guard": float(ratio.mean().detach().cpu()),
        "row_ratio_max_before_guard": float(ratio.max().detach().cpu()),
    }


def _factorize_balanced_from_basis(U_basis: torch.Tensor, core: torch.Tensor, V_basis: torch.Tensor):
    """
    Factor a projected rank-k core into LoRA B/A factors.

    U_basis: [out_dim, k] orthonormal columns
    core:    [k, k] projected dense delta
    V_basis: [in_dim, k] orthonormal columns

    Produces:
      B_new: [out_dim, k]
      A_new: [k, in_dim]
    """
    P, Sc, Qh = torch.linalg.svd(core.float(), full_matrices=False)
    Sc = Sc.float().clamp_min(0.0)
    sroot = torch.sqrt(Sc.clamp_min(1e-12))
    B_new = (U_basis.float() @ P.float()) * sroot.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * (Qh.float() @ V_basis.float().T)
    return B_new.contiguous(), A_new.contiguous(), Sc.contiguous()

def _rebuild_objective(B_new: torch.Tensor, A_new: torch.Tensor, delta: torch.Tensor):
    """
    Deterministic accept metric.

    Primary term is global residual. Secondary term penalizes row overshoot.
    This keeps the rebuild from winning merely by becoming louder.
    """
    recon = B_new.float() @ A_new.float()
    delta_norm = torch.linalg.vector_norm(delta.float()).clamp_min(1e-12)
    residual = torch.linalg.vector_norm((delta.float() - recon).float()) / delta_norm

    orig_row = torch.linalg.vector_norm(delta.float(), ord=2, dim=1).clamp_min(1e-12)
    new_row = torch.linalg.vector_norm(recon.float(), ord=2, dim=1).clamp_min(1e-12)
    row_ratio = new_row / orig_row
    overshoot = torch.clamp(row_ratio - float(ROW_NORM_GAIN_CAP), min=0.0).mean()

    return residual + 0.05 * overshoot, residual, overshoot

def _deterministic_subspace_rebuild(
    U_k: torch.Tensor,
    Vh_k: torch.Tensor,
    S_k: torch.Tensor,
    gain_vec: torch.Tensor,
    delta: torch.Tensor,
    baseline_B: torch.Tensor,
    baseline_A: torch.Tensor,
    baseline_row_guard_stats: dict,
):
    """
    Deterministic lost-information rebuild inside the same rank contract.

    PairFold changes the survivor directions, but the old diagonal factorization
    still only uses one singular lane at a time. This rebuild treats the survivor
    left/right directions as a rank-k subspace and computes the best k x k core
    projection of the original dense delta inside that subspace:

        core = Q_left.T @ Delta @ Q_right

    That core contains cross-lane residual information that the diagonal path
    discards. We then SVD-factor the core back into standard LoRA B/A tensors.
    No extra rank is created, no randomness is used, and the output still obeys
    the same `(out_dim, rank)` / `(rank, in_dim)` evaluator contract.
    """
    if not DETERMINISTIC_REBUILD_ENABLED:
        return baseline_B.contiguous(), baseline_A.contiguous(), {
            "deterministic_rebuild_enabled": False,
            "deterministic_rebuild_mode": "disabled",
        }, baseline_row_guard_stats

    if U_k.numel() == 0 or Vh_k.numel() == 0:
        return baseline_B.contiguous(), baseline_A.contiguous(), {
            "deterministic_rebuild_enabled": True,
            "deterministic_rebuild_mode": "empty_basis_fallback",
        }, baseline_row_guard_stats

    with torch.no_grad():
        base_obj, base_resid, base_overshoot = _rebuild_objective(baseline_B, baseline_A, delta)

        # Build orthonormal survivor bases. QR gives deterministic bases for a
        # fixed input matrix and avoids relying on non-orthogonal blended lanes.
        Q_left, _ = torch.linalg.qr(U_k.float(), mode="reduced")
        Q_right, _ = torch.linalg.qr(Vh_k.float().T, mode="reduced")

        core = Q_left.T @ delta.float() @ Q_right
        B_core, A_core, Sc_core = _factorize_balanced_from_basis(Q_left, core, Q_right)

        # Keep total restored energy aligned to the existing cap/gain target.
        target_energy = torch.sqrt(torch.sum((S_k.float() * gain_vec.float()) ** 2)).clamp_min(1e-12)
        core_energy = torch.sqrt(torch.sum(Sc_core.float() ** 2)).clamp_min(1e-12)
        energy_scale = (target_energy / core_energy).clamp(0.25, 4.0)
        if torch.isfinite(energy_scale):
            scale_root = torch.sqrt(energy_scale)
            B_core = B_core * scale_root
            A_core = A_core * scale_root

        B_core, rebuild_row_guard_stats = _apply_row_norm_guard(B_core, A_core, delta)
        cand_obj, cand_resid, cand_overshoot = _rebuild_objective(B_core, A_core, delta)

        accept = bool(cand_obj <= base_obj * (1.0 + float(DETERMINISTIC_REBUILD_ACCEPT_EPS)))

    stats = {
        "deterministic_rebuild_enabled": True,
        "deterministic_rebuild_mode": "accepted_subspace_core_projection" if accept else "rejected_kept_pairfold_diagonal",
        "deterministic_rebuild_accepted": bool(accept),
        "deterministic_rebuild_base_objective": float(base_obj.detach().cpu()),
        "deterministic_rebuild_candidate_objective": float(cand_obj.detach().cpu()),
        "deterministic_rebuild_base_residual": float(base_resid.detach().cpu()),
        "deterministic_rebuild_candidate_residual": float(cand_resid.detach().cpu()),
        "deterministic_rebuild_base_overshoot": float(base_overshoot.detach().cpu()),
        "deterministic_rebuild_candidate_overshoot": float(cand_overshoot.detach().cpu()),
        "deterministic_rebuild_energy_scale": float(energy_scale.detach().cpu()),
        "deterministic_rebuild_core_energy": float(core_energy.detach().cpu()),
        "deterministic_rebuild_target_energy": float(target_energy.detach().cpu()),
    }

    if accept:
        return B_core.contiguous(), A_core.contiguous(), stats, rebuild_row_guard_stats
    return baseline_B.contiguous(), baseline_A.contiguous(), stats, baseline_row_guard_stats


def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int, component_slices=None):
    """
    Compress Delta = B @ A to rank-k with PairFold residual matching.

    The output shape is always `(out_dim, rank)` and `(rank, in_dim)` unless
    the underlying matrix is smaller than the requested rank.
    """
    if B.ndim != 2 or A_mat.ndim != 2:
        raise ValueError(f"Expected 2D LoRA matrices, got B={tuple(B.shape)}, A={tuple(A_mat.shape)}")
    if B.shape[1] != A_mat.shape[0]:
        raise ValueError(f"LoRA inner rank mismatch: B={tuple(B.shape)}, A={tuple(A_mat.shape)}")

    delta = B.float() @ A_mat.float()
    if not torch.isfinite(delta).all():
        raise FloatingPointError("Dense LoRA delta contains non-finite values before compression")

    max_rank = min(delta.shape)
    rank = min(int(rank), int(max_rank))
    if rank <= 0:
        raise ValueError(f"Invalid compression rank {rank} for delta shape {tuple(delta.shape)}")

    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    total_mass = S.sum().clamp_min(1e-12)
    full_energy = torch.sqrt(torch.sum(S ** 2)).clamp_min(1e-12)

    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]

    kept_energy = torch.sqrt(torch.sum(S_k ** 2)).clamp_min(1e-12)
    raw_gain = full_energy / kept_energy
    gain_vec, gain_stats = _dual_lane_gain_vector(
        singular_values=S_k,
        raw_gain=raw_gain,
        rank=rank,
    )

    row_blocks = _make_row_blocks(int(delta.shape[0]), component_slices=component_slices)
    U_k, Vh_k, gain_vec, pairfold_stats = _pairfold_transport(
        U=U,
        S=S,
        Vh=Vh,
        rank=rank,
        gain_vec=gain_vec,
        row_blocks=row_blocks,
    )

    # Baseline balanced transport: split scale symmetrically across both LoRA factors.
    sroot_balanced = torch.sqrt(S_k * gain_vec)
    B_baseline = U_k * sroot_balanced.unsqueeze(0)
    A_baseline = sroot_balanced.unsqueeze(1) * Vh_k
    B_baseline, row_guard_stats = _apply_row_norm_guard(B_baseline, A_baseline, delta)

    # Deterministic rebuild: recover residual cross-lane information inside the
    # same rank-32 evaluator contract, then accept only if the local objective
    # does not regress.
    B_new, A_new, rebuild_stats, row_guard_stats = _deterministic_subspace_rebuild(
        U_k=U_k,
        Vh_k=Vh_k,
        S_k=S_k,
        gain_vec=gain_vec,
        delta=delta,
        baseline_B=B_baseline,
        baseline_A=A_baseline,
        baseline_row_guard_stats=row_guard_stats,
    )

    restored_energy = torch.sqrt(torch.sum((S_k * gain_vec) ** 2)).clamp_min(1e-12)
    recon = B_new.float() @ A_new.float()
    residual_energy = torch.linalg.vector_norm((delta - recon).float()).clamp_min(1e-12)

    if not torch.isfinite(B_new).all() or not torch.isfinite(A_new).all():
        raise FloatingPointError("Compressed LoRA factors contain non-finite values")

    stats = {
        "rank_in": int(B.shape[1]),
        "rank_out": int(rank),
        "delta_shape": [int(delta.shape[0]), int(delta.shape[1])],
        "singular_mass_kept": float((S_k.sum() / total_mass).detach().cpu()),
        "energy_kept_ratio": float((kept_energy / full_energy).detach().cpu()),
        "energy_after_gain_ratio": float((restored_energy / full_energy).detach().cpu()),
        "reconstruction_residual_ratio": float((residual_energy / torch.linalg.vector_norm(delta).clamp_min(1e-12)).detach().cpu()),
        "gain_distribution": GAIN_DISTRIBUTION,
        "row_block_count": int(len(row_blocks)),
        "row_blocks": [{"name": name, "rows": [int(start), int(end)]} for start, end, name in row_blocks],
        **gain_stats,
        **pairfold_stats,
        **rebuild_stats,
        **row_guard_stats,
    }

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous(), stats

# -----------------------------------------------------------------------------
# Tinker patch: fused projection merge
# -----------------------------------------------------------------------------
def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    if component_order is None:
        raise RuntimeError(f"No fused projection component order found for {fused_target_name!r}")

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(f"Missing component {comp_name!r} for fused target {fused_model_key!r}")

        lora_A, lora_B = comp_by_name[comp_name]
        if lora_A.ndim != 2 or lora_B.ndim != 2:
            raise ValueError(f"Expected 2D LoRA tensors for {comp_name}: A={tuple(lora_A.shape)}, B={tuple(lora_B.shape)}")
        if lora_A.shape[0] != lora_B.shape[1]:
            raise ValueError(f"Component rank mismatch for {comp_name}: A={tuple(lora_A.shape)}, B={tuple(lora_B.shape)}")

        r = int(lora_A.shape[0])
        out_dim = int(lora_B.shape[0])

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r, comp_name))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim,
        merged_rank,
        dtype=merged_lora_A.dtype,
        device=merged_lora_A.device,
    )

    rank_offset = 0
    for row_start, row_end, r, comp_name in comp_slices:
        _, lora_B = comp_by_name[comp_name]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    final_rank = int(merged_rank)
    compression_stats = {
        "rank_in": int(merged_rank),
        "rank_out": int(merged_rank),
        "preservation": "exact_no_compression",
        "component_row_total": int(row_offset),
        "fused_out_dim": int(fused_out_dim),
    }

    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A, svd_stats = _compress_lora_pair_to_rank(
            merged_lora_B,
            merged_lora_A,
            FORCED_FUSED_RANK,
            component_slices=comp_slices,
        )
        final_rank = int(merged_lora_A.shape[0])
        compression_stats = {
            **svd_stats,
            "preservation": "rank32_pairfold_deterministic_rebuild_rowguard",
            "gain_cap": float(SVD_ENERGY_GAIN_CAP),
        }

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"

    GLYPH_LEDGER.emit(
        src=f"{adapter_layer_prefix}.{{{','.join(component_order)}}}",
        dst=peft_target_key,
        op="fused_projection_pairfold_transport",
        alpha="mamba_or_fused_projection",
        beta={
            "fused_model_key": fused_model_key,
            "fused_out_dim": int(fused_out_dim),
            "component_count": len(component_order),
            "component_order": list(component_order),
            "component_slices": [
                {"name": name, "rows": [int(a), int(b)], "rank": int(r)}
                for a, b, r, name in comp_slices
            ],
        },
        gamma=compression_stats,
    )

    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank

if not hasattr(A, "_merge_fused_projections"):
    raise RuntimeError("tinker_cookbook.weights._adapter._merge_fused_projections not found")

A._merge_fused_projections = patched_merge_fused_projections

print("[GlyphMatics] patched:", A._merge_fused_projections.__name__)
print("[GlyphMatics] FORCED_FUSED_RANK:", FORCED_FUSED_RANK)
print("[GlyphMatics] SVD_ENERGY_GAIN_CAP:", SVD_ENERGY_GAIN_CAP)
print("[GlyphMatics] GAIN_DISTRIBUTION:", GAIN_DISTRIBUTION)
print("[GlyphMatics] DETERMINISTIC_REBUILD_ENABLED:", DETERMINISTIC_REBUILD_ENABLED)
print("[GlyphMatics] DETERMINISTIC_REBUILD_ACCEPT_EPS:", DETERMINISTIC_REBUILD_ACCEPT_EPS)
print("[GlyphMatics] PAIRFOLD_ENABLED:", PAIRFOLD_ENABLED)
print("[GlyphMatics] PAIRFOLD_PAIR_WIDTH:", PAIRFOLD_PAIR_WIDTH)
print("[GlyphMatics] PAIRFOLD_TAIL_MAX:", PAIRFOLD_TAIL_MAX)
print("[GlyphMatics] PAIRFOLD_MIN_SIM:", PAIRFOLD_MIN_SIM, "# Variant C strict tail")
print("[GlyphMatics] PAIRFOLD_MAX_GAIN_DELTA:", PAIRFOLD_MAX_GAIN_DELTA)
print("[GlyphMatics] PAIRFOLD_VECTOR_BLEND:", PAIRFOLD_VECTOR_BLEND)
print("[GlyphMatics] ROW_NORM_GUARD_ENABLED:", ROW_NORM_GUARD_ENABLED)
print("[GlyphMatics] ROW_NORM_GAIN_CAP:", ROW_NORM_GAIN_CAP)
print("[GlyphMatics] DUAL_PAIR_ENABLED:", DUAL_PAIR_ENABLED)
print("[GlyphMatics] DUAL_PAIR_SPLIT:", DUAL_PAIR_SPLIT)


[GlyphMatics] patched: patched_merge_fused_projections
[GlyphMatics] FORCED_FUSED_RANK: 32
[GlyphMatics] SVD_ENERGY_GAIN_CAP: 1.19
[GlyphMatics] GAIN_DISTRIBUTION: pairfold_residual_matched_deterministic_rebuild_rowguard
[GlyphMatics] DETERMINISTIC_REBUILD_ENABLED: True
[GlyphMatics] DETERMINISTIC_REBUILD_ACCEPT_EPS: 0.0
[GlyphMatics] PAIRFOLD_ENABLED: True
[GlyphMatics] PAIRFOLD_PAIR_WIDTH: 2
[GlyphMatics] PAIRFOLD_TAIL_MAX: 96
[GlyphMatics] PAIRFOLD_MIN_SIM: 0.7 # Variant C strict tail
[GlyphMatics] PAIRFOLD_MAX_GAIN_DELTA: 0.022
[GlyphMatics] PAIRFOLD_VECTOR_BLEND: 0.025
[GlyphMatics] ROW_NORM_GUARD_ENABLED: True
[GlyphMatics] ROW_NORM_GAIN_CAP: 1.08
[GlyphMatics] DUAL_PAIR_ENABLED: False
[GlyphMatics] DUAL_PAIR_SPLIT: [0.62, 0.58]


## 4. Local compression self-tests

These tests run before building the adapter. They validate the new PairFold path using synthetic LoRA tensors, without requiring the full model weights to be loaded.


In [8]:
import torch

print("[SelfTest] deterministic PairFold compression tests")
_gen = torch.Generator(device="cpu").manual_seed(918)

B_test = torch.randn(48, 24, generator=_gen, dtype=torch.float32)
A_test = torch.randn(24, 40, generator=_gen, dtype=torch.float32)
component_slices_test = [
    (0, 12, 8, "q_proj"),
    (12, 24, 8, "k_proj"),
    (24, 36, 4, "v_proj"),
    (36, 48, 4, "gate_proj"),
]

B_comp, A_comp, stats = _compress_lora_pair_to_rank(
    B_test,
    A_test,
    rank=8,
    component_slices=component_slices_test,
)

assert tuple(B_comp.shape) == (48, 8), tuple(B_comp.shape)
assert tuple(A_comp.shape) == (8, 40), tuple(A_comp.shape)
assert torch.isfinite(B_comp).all(), "B_comp contains non-finite values"
assert torch.isfinite(A_comp).all(), "A_comp contains non-finite values"

orig_delta = B_test @ A_test
new_delta = B_comp.float() @ A_comp.float()
assert torch.isfinite(new_delta).all(), "reconstructed delta contains non-finite values"

if ROW_NORM_GUARD_ENABLED:
    orig_row = torch.linalg.vector_norm(orig_delta, dim=1).clamp_min(1e-12)
    new_row = torch.linalg.vector_norm(new_delta, dim=1).clamp_min(1e-12)
    worst_ratio = float((new_row / orig_row).max().detach().cpu())
    assert worst_ratio <= ROW_NORM_GAIN_CAP + 2e-4, (worst_ratio, ROW_NORM_GAIN_CAP)

assert stats["rank_out"] == 8, stats
assert stats["pairfold_enabled"] == bool(PAIRFOLD_ENABLED), stats
assert 0.0 <= stats["energy_kept_ratio"] <= 1.000001, stats
assert stats["reconstruction_residual_ratio"] >= 0.0, stats

print("[SelfTest] passed")
print(json.dumps({
    "rank_in": stats["rank_in"],
    "rank_out": stats["rank_out"],
    "energy_kept_ratio": stats["energy_kept_ratio"],
    "energy_after_gain_ratio": stats["energy_after_gain_ratio"],
    "pairfold_mode": stats.get("pairfold_mode"),
    "pairfold_assignments": stats.get("pairfold_assignments"),
    "row_clip_fraction": stats.get("row_clip_fraction"),
    "reconstruction_residual_ratio": stats["reconstruction_residual_ratio"],
}, indent=2))

assert stats.get("deterministic_rebuild_enabled") is True, "Deterministic rebuild flag missing"
assert stats.get("deterministic_rebuild_mode") in {
    "accepted_subspace_core_projection",
    "rejected_kept_pairfold_diagonal",
    "empty_basis_fallback",
}, stats.get("deterministic_rebuild_mode")
print("[SelfTest] deterministic rebuild mode:", stats.get("deterministic_rebuild_mode"))
print("[SelfTest] deterministic rebuild accepted:", stats.get("deterministic_rebuild_accepted"))


[SelfTest] deterministic PairFold compression tests
[SelfTest] passed
{
  "rank_in": 24,
  "rank_out": 8,
  "energy_kept_ratio": 0.8670838475227356,
  "energy_after_gain_ratio": 1.0,
  "pairfold_mode": "residual_matched_rank_pairs",
  "pairfold_assignments": 32,
  "row_clip_fraction": 0.1458333283662796,
  "reconstruction_residual_ratio": 0.5145506262779236
}
[SelfTest] deterministic rebuild mode: accepted_subspace_core_projection
[SelfTest] deterministic rebuild accepted: True


## 5. Build adapter package

This clears stale output, builds the converted adapter, and creates required marker files.


In [9]:
from pathlib import Path
import shutil
import json
import time
from tinker_cookbook import weights

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

if OUTPUT_DIR.exists():
    print("[Build] removing stale output:", OUTPUT_DIR)
    shutil.rmtree(OUTPUT_DIR)

print("[Build] adapter_path:", ADAPTER_PATH)
print("[Build] base_model_path:", BASE_MODEL_PATH)
print("[Build] output_dir:", OUTPUT_DIR)

t0 = time.time()
weights.build_lora_adapter(
    base_model=str(BASE_MODEL_PATH),
    adapter_path=str(ADAPTER_PATH),
    output_path=str(OUTPUT_DIR),
)
elapsed = time.time() - t0
print(f"[Build] build_lora_adapter completed in {elapsed/60:.2f} min")

required_core = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

missing_core = [name for name in required_core if not (OUTPUT_DIR / name).exists()]
if missing_core:
    raise FileNotFoundError(f"Adapter build did not produce required files: {missing_core}")

# Required by some submit/eval paths.
(OUTPUT_DIR / "checkpoint_complete").write_text("ok\n", encoding="utf-8")

GLYPH_LEDGER.print_summary()
ledger_text = GLYPH_LEDGER.markdown()
(OUTPUT_DIR / "GLYPHMATICS_TRANSPORT_LEDGER.md").write_text(ledger_text, encoding="utf-8")

repair_summary = globals().get("REPAIR_TRACE_SUMMARY", {})
repair_trace_path = globals().get("REPAIR_TRACE_PATH", None)
crypto_summary = globals().get("CRYPTO_TRACE_SUMMARY", {})
crypto_trace_path = globals().get("CRYPTO_REPAIR_TRACE_PATH", None)
crypto_v2_path = globals().get("CRYPTO_V2_CSV_PATH", None)

readme_text = [
    "# Nemotron Adapter Submission",
    "",
    "GlyphMatics v9D4: strict-tail PairFold residual-matched row-guarded fused-projection transport plus substitution-cipher and solver-verified cryptarithm repair-gate audits.",
    "",
    "## Build knobs",
    "",
    f"- FORCED_FUSED_RANK: {FORCED_FUSED_RANK}",
    f"- SVD_ENERGY_GAIN_CAP: {SVD_ENERGY_GAIN_CAP}",
    f"- GAIN_DISTRIBUTION: {GAIN_DISTRIBUTION}",
    f"- PAIRFOLD_ENABLED: {PAIRFOLD_ENABLED}",
    f"- PAIRFOLD_PAIR_WIDTH: {PAIRFOLD_PAIR_WIDTH}",
    f"- PAIRFOLD_TAIL_MAX: {PAIRFOLD_TAIL_MAX}",
    f"- PAIRFOLD_MIN_SIM: {PAIRFOLD_MIN_SIM}",
    f"- PAIRFOLD_MAX_GAIN_DELTA: {PAIRFOLD_MAX_GAIN_DELTA}",
    f"- PAIRFOLD_VECTOR_BLEND: {PAIRFOLD_VECTOR_BLEND}",
    f"- ROW_NORM_GUARD_ENABLED: {ROW_NORM_GUARD_ENABLED}",
    f"- ROW_NORM_GAIN_CAP: {ROW_NORM_GAIN_CAP}",
    f"- DUAL_PAIR_ENABLED: {DUAL_PAIR_ENABLED}",
    f"- DUAL_PAIR_SPLIT: {DUAL_PAIR_SPLIT}",
    "",
    "## D4 repair-gate audit",
    "",
    f"- substitution_status: {repair_summary.get('status', 'missing')}",
    f"- substitution_train_accuracy: {repair_summary.get('accuracy', 'n/a')}",
    f"- substitution_compact_traces: {repair_summary.get('compact_traces', 0)}",
    f"- substitution_rejected_conflicts: {repair_summary.get('rejected_conflicts', 0)}",
    f"- substitution_trace_path: {repair_trace_path}",
    f"- cryptarithm_status: {crypto_summary.get('status', 'missing')}",
    f"- cryptarithm_verified_exact: {crypto_summary.get('verified_exact', 0)}",
    f"- cryptarithm_scanned_rows: {crypto_summary.get('scanned_rows', 0)}",
    f"- cryptarithm_trace_path: {crypto_trace_path}",
    f"- cryptarithm_v2_csv_path: {crypto_v2_path}",
    "",
]
readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    existing = readme_path.read_text(encoding="utf-8", errors="replace")
    readme_path.write_text(existing.rstrip() + "\n\n" + "\n".join(readme_text) + "\n", encoding="utf-8")
else:
    readme_path.write_text("\n".join(readme_text) + "\n", encoding="utf-8")

manifest = {
    "name": "glyphmatics_nemotron_v9d4_subcipher_crypto_gate_competition_ready",
    "output_dir": str(OUTPUT_DIR),
    "elapsed_seconds": elapsed,
    "forced_fused_rank": FORCED_FUSED_RANK,
    "svd_energy_gain_cap": SVD_ENERGY_GAIN_CAP,
    "gain_distribution": GAIN_DISTRIBUTION,
    "pairfold_enabled": PAIRFOLD_ENABLED,
    "pairfold_pair_width": PAIRFOLD_PAIR_WIDTH,
    "pairfold_tail_max": PAIRFOLD_TAIL_MAX,
    "pairfold_min_sim": PAIRFOLD_MIN_SIM,
    "pairfold_max_gain_delta": PAIRFOLD_MAX_GAIN_DELTA,
    "pairfold_vector_blend": PAIRFOLD_VECTOR_BLEND,
    "row_norm_gain_cap": ROW_NORM_GAIN_CAP,
    "dual_pair_enabled": DUAL_PAIR_ENABLED,
    "dual_pair_split": DUAL_PAIR_SPLIT,
    "ledger_events": len(GLYPH_LEDGER.events),
    "d3_repair_trace_summary": repair_summary,
    "d4_cryptarithm_trace_summary": crypto_summary,
}
(OUTPUT_DIR / "submission_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("[Build] output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size)


[Build] adapter_path: /kaggle/input/models/huikang/nemotron-adapter/transformers/default/20
[Build] base_model_path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
[Build] output_dir: /kaggle/working/nemotron-adapter-ready-to-submit


MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


[Build] build_lora_adapter completed in 3.22 min
[GlyphMatics ledger] events: 23
[GlyphMatics ledger] mamba_or_fused_projection:fused_projection_pairfold_transport=23
[Build] output files:
 - GLYPHMATICS_TRANSPORT_LEDGER.md 59318
 - README.md 1186
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 3
 - submission_manifest.json 2117


## 5A. Attach NGSC v3 summary to adapter README

Keeps `submission.zip` member list strict while preserving NGSC provenance in `README.md`.

In [10]:
# Append NGSC v3 manifest details into the adapter output README without changing required zip member list.
from pathlib import Path
import json, hashlib

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
NGSC_PATH = Path("/kaggle/working/nemotron_glyph_cipher_v3.json")

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"OUTPUT_DIR missing before NGSC augmentation: {OUTPUT_DIR}")
if not NGSC_PATH.exists():
    raise FileNotFoundError(f"NGSC manifest missing: {NGSC_PATH}")

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

ngsc = json.loads(NGSC_PATH.read_text(encoding="utf-8"))
ngsc_sha = sha256_file(NGSC_PATH)
ngsc_summary = {
    "ngsc_name": ngsc.get("meta", {}).get("name"),
    "segments": len(ngsc.get("segment_decode", {})),
    "glyphs": len(ngsc.get("glyph_registry", {})),
    "modifiers": len(ngsc.get("modifier_registry", {})),
    "operators": len(ngsc.get("operator_registry", {})),
    "sha256": ngsc_sha,
}

(OUTPUT_DIR / "NGSC_V3_MANIFEST_SUMMARY.json").write_text(
    json.dumps(ngsc_summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

readme = OUTPUT_DIR / "README.md"
appendix = """

## NGSC v3 substitution-cipher manifest

- name: {name}
- manifest_path: {path}
- sha256: {sha}
- segments: {segments}
- glyphs: {glyphs}
- modifiers: {modifiers}
- operators: {operators}
""".format(
    name=ngsc_summary["ngsc_name"],
    path=str(NGSC_PATH),
    sha=ngsc_summary["sha256"],
    segments=ngsc_summary["segments"],
    glyphs=ngsc_summary["glyphs"],
    modifiers=ngsc_summary["modifiers"],
    operators=ngsc_summary["operators"],
)
readme.write_text(readme.read_text(encoding="utf-8", errors="replace").rstrip() + appendix + "\n", encoding="utf-8")
print("[NGSC] README augmented")
print(json.dumps(ngsc_summary, indent=2, ensure_ascii=False))


[NGSC] README augmented
{
  "ngsc_name": "Nemotron Glyph Substitution Cipher v3.0",
  "segments": 38,
  "glyphs": 16,
  "modifiers": 16,
  "operators": 16,
  "sha256": "388da2f88b27e9ddd74a1bb09d3e963f10fc96eca9cdec46b92b6fcd5a4232bb"
}


## 6. Validate and create `submission.zip`

The zip is intentionally minimal: only the files most likely required by the evaluator are included.


In [11]:
import zipfile
import hashlib
from pathlib import Path
import json

OUTPUT_DIR = Path("/kaggle/working/nemotron-adapter-ready-to-submit")
ZIP_PATH = Path("/kaggle/working/submission.zip")

required = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "README.md",
    "checkpoint_complete",
]

missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required submission files: {missing}")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in required:
        p = OUTPUT_DIR / name
        zf.write(p, arcname=name)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    bad = zf.testzip()

print("[Zip] wrote:", ZIP_PATH)
print("[Zip] size:", ZIP_PATH.stat().st_size)
print("[Zip] sha256:", sha256_file(ZIP_PATH))
print("[Zip] contents:")
for name in names:
    print(" -", name)

if bad is not None:
    raise RuntimeError(f"Zip integrity check failed at member: {bad}")
if ZIP_PATH.stat().st_size <= 0:
    raise RuntimeError("submission.zip is empty")
if names != required:
    raise RuntimeError(f"Unexpected zip contents/order: {names}")

print("[Zip] validation passed")


[Zip] wrote: /kaggle/working/submission.zip
[Zip] size: 3270301359
[Zip] sha256: 3308fb74ce555774b1a93b27b817187500b6675e3ba124a859240be4da765b1e
[Zip] contents:
 - adapter_config.json
 - adapter_model.safetensors
 - README.md
 - checkpoint_complete
[Zip] validation passed


## 7. Final listing

Submit `/kaggle/working/submission.zip`.


In [12]:
from pathlib import Path
import os

for p in [
    Path("/kaggle/working/submission.zip"),
    Path("/kaggle/working/nemotron-adapter-ready-to-submit"),
]:
    print("\n[Listing]", p)
    if p.is_file():
        print(p, p.stat().st_size)
    elif p.is_dir():
        for child in sorted(p.iterdir()):
            print(" -", child.name, child.stat().st_size)
    else:
        print("MISSING:", p)



[Listing] /kaggle/working/submission.zip
/kaggle/working/submission.zip 3270301359

[Listing] /kaggle/working/nemotron-adapter-ready-to-submit
 - GLYPHMATICS_TRANSPORT_LEDGER.md 59318
 - NGSC_V3_MANIFEST_SUMMARY.json 213
 - README.md 1474
 - adapter_config.json 618
 - adapter_model.safetensors 3554384888
 - checkpoint_complete 3
 - submission_manifest.json 2117


## 8. Final hard verification

This is the last guard before pressing **Submit Prediction**. It fails if the notebook did not produce a non-empty `submission.zip` with the required adapter members.


In [13]:
from pathlib import Path
import zipfile, hashlib

SUB = Path("/kaggle/working/submission.zip")
print("[FinalVerify] exists:", SUB.exists())
if not SUB.exists():
    raise FileNotFoundError("/kaggle/working/submission.zip was not created")

size = SUB.stat().st_size
print("[FinalVerify] size_bytes:", size)
if size <= 0:
    raise RuntimeError("submission.zip is empty")

with zipfile.ZipFile(SUB, "r") as z:
    names = z.namelist()
    bad = z.testzip()

print("[FinalVerify] zip_names:", names)
required_members = {"adapter_config.json", "adapter_model.safetensors", "README.md", "checkpoint_complete"}
missing = sorted(required_members - set(names))
extra = sorted(set(names) - required_members)
if bad is not None:
    raise RuntimeError(f"zip integrity failure at {bad}")
if missing:
    raise RuntimeError(f"missing required zip members: {missing}")
if extra:
    raise RuntimeError(f"unexpected zip members: {extra}")

h = hashlib.sha256()
with SUB.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)
print("[FinalVerify] sha256:", h.hexdigest())
print("[FinalVerify] PASS: submit /kaggle/working/submission.zip")


[FinalVerify] exists: True
[FinalVerify] size_bytes: 3270301359
[FinalVerify] zip_names: ['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'checkpoint_complete']
[FinalVerify] sha256: 3308fb74ce555774b1a93b27b817187500b6675e3ba124a859240be4da765b1e
[FinalVerify] PASS: submit /kaggle/working/submission.zip
